In [ ]:
%load_ext autoreload
%aimport pyhipp
%aimport baryon_cycle
%aimport

In [ ]:
from typing import Any
from pyhipp import plot, stats
from pyhipp_sims import sims
from pyhipp.core import DataDict, Num
from baryon_cycle.config import ProjPaths, ColorSets, c_sets, cs_named
from pyhipp.astro.coords import cvt as coords_cvt
from pathlib import Path
from pyhipp.io import h5, json
import numpy as np
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from dataclasses import dataclass
from scipy.optimize import minimize
from astropy.io import fits
from pyhipp.stats import Rng
from baryon_cycle.ecosys_info import EcosysInfo
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

In [ ]:
# plot_data_dir = ProjPaths.proj_dir / 'plots_for_pub/data'
sims_dir = ProjPaths.sims_dir
sim_dir = sims_dir / "tng_50_1"
out_fig_dir = ProjPaths.figs_dir
sim_info = sims.predefined['tng_50_1']

#### Global properties
ecosys_defs: DataDict[str, DataDict[str, DataDict]] = json.File.load_file(sim_dir/"ecosys_defs.json")
ecosys_infos: DataDict[str, EcosysInfo] = DataDict()
for k, d in ecosys_defs.items():
    ecosys_infos[k] = DataDict()
    for k1, d1 in d.items():
        info = EcosysInfo.from_dict(d1)
        ecosys_infos[k][k1] = info
        d1 |= info.to_dict()
        
ecosys_infos['z0'].keys()
ecosys_defs['z0/lm10']

panel_number_kw = dict(fontfamily='Arial', fontdict={'weight':'bold'}, fontsize=17)

# Slightline Integration

In [ ]:
%autoreload

from baryon_cycle.sightlines import SightlineIntegrator, TiltedDiskLocalFrame
from pyhipp.io import json
class Lan2018Fig4:
    def __init__(self):
        f_in = ProjPaths.obs_dir / 'Lan2018/Fig4.json'
        d_in = json.File.load_file(f_in)['left']
        d_out = DataDict()
        for k in 'polar', 'disk':
            d = d_in[k]
            x, y, y_lo, y_hi = d[0::3,0], d[0::3,1], d[1::3,1], d[2::3,1]
            lx, ly = np.log10(x), np.log10(y)
            ly_lo, ly_hi = np.log10(y_lo), np.log10(y_hi)
            e_lo, e_hi = y - y_lo, y_hi - y
            le_lo, le_hi = ly - ly_lo, ly_hi - ly
            d_out[k] = DataDict({
                'x': x, 'y': y, 'e_lo': e_lo, 'e_hi': e_hi,
                'y_lo': y_lo, 'y_hi': y_hi,
                'lx': lx, 'ly': ly, 'le_lo': le_lo, 'le_hi': le_hi,
                'ly_lo': ly_lo, 'ly_hi': ly_hi,
            })
        self.data = d_out
        
    def ratio_data(self):
        d = self.data
        lx1, ly1, lel1, leh1 = d['polar']['lx', 'ly', 'le_lo', 'le_hi']
        lx2, ly2, lel2, leh2 = d['disk']['lx', 'ly', 'le_lo', 'le_hi']
        
        ly1 = np.interp(lx2, lx1, ly1)
        lel1 = np.interp(lx2, lx1, lel1)
        leh1 = np.interp(lx2, lx1, leh1)
        out = []
        rng = stats.Rng(10086)
        for row in np.column_stack([ly1, lel1, leh1, ly2, lel2, leh2]):
            diff, dl, dh = self.diff_with_err(*row, rng=rng)
            r, rl, rh = 10.0**diff, 10.0**dl, 10.0**dh
            el, eh = r - rl, rh - r
            out.append((r, el, eh))
        r, el, eh = np.array(out).T
        return DataDict({
            'x': 10.0**lx2, 
            'r': r, 'e_lo': el, 'e_hi': eh,
        })
        
    def diff_with_err(self, y1, el1, eh1, y2, el2, eh2, rng: stats.Rng,
                       n_samp=1024):
        y1s = self.samp(y1, el1, eh1, rng, n_samp)
        y2s = self.samp(y2, el2, eh2, rng, n_samp)
        diffs = y1s - y2s
        diff, (dl, dh) = np.median(diffs), np.quantile(diffs, [0.16, 0.84])
        return diff, dl, dh

    def samp(self, y, el, eh, rng: stats.Rng, n_samp=1024):
        rv_n = rng.standard_normal(size=n_samp)
        neg = rv_n < 0
        rv_n[neg] *= el
        rv_n[~neg] *= eh
        return y + rv_n

lan_2018_data = Lan2018Fig4()
lan_2018_ratio = lan_2018_data.ratio_data()

In [ ]:
f_in = sim_dir / '2Dmaps_individual_interp.hdf5'
fields_2D = h5.File.load_from(f_in, key='z0/lm10')  # sample
header = h5.File.load_from(f_in, key='header')

r_es, theta_es, phi_es = header['r_edges', 'theta_edges', 'phi_edges']
theta_es = theta_es[::-1]       # to be in ascending order

us = sim_info.unit_system
u_kpc2cm = us.astropy_u.kpc.to('cm')
u_kpc2cm_3 = u_kpc2cm**3
u_kpc2cm_2 = u_kpc2cm**2

In [ ]:
#vol_dens = fields_2D['n_O']  * u_kpc2cm_3     # kpc^-3
vol_dens = fields_2D['n_Mg']  * u_kpc2cm_3     # kpc^-3
print(f'vol_dens shape={vol_dens.shape}')

n_phis = len(phi_es) - 1
n_thetas = len(theta_es) - 1
n_rs = len(r_es) - 1
n_gals = len(vol_dens) // n_phis

n1 = vol_dens.reshape((n_gals, n_phis, n_rs, n_thetas))
print(f'n1 shape={n1.shape}')

n2 = np.transpose(n1, (0, 2, 3, 1))
n2 = n2[:,:,::-1,:]
print(f'n2 shape={n2.shape}')

n_med = np.median(n2, axis=(0,))

In [ ]:
rng = Rng(10086)
n_sample = 50000
gids = rng.choice(n_gals, size=n_sample)

outs = []
for i, gid in enumerate(gids):
    lf = TiltedDiskLocalFrame.new_random(rng)
    integ = SightlineIntegrator(lf, 
                                n2[gid], 
                                #n_med,
                                r_es, theta_es, phi_es)
    r_p, ang = np.linalg.norm(lf.r_p), lf.angle_r_p2ez_p()
    col_dens = integ.integrate(z_max=1500.0, 
                               eps=.5) / u_kpc2cm_2
    outs.append((r_p, ang, col_dens))
    
r_ps, angs, col_denss = np.array(outs).T

#fout = sim_dir/'metal'/'z0-lm10-individual.hdf5'
#fout = sim_dir/'metal'/'z0-lm10-median.hdf5'
#fout = sim_dir/'metal'/'z1-lm10-median.hdf5'
# fout = sim_dir/'metal'/'z1-lm10-individual.hdf5'
fout = sim_dir/'metal'/'z0-lm10-individual-Mg.hdf5'
#fout = sim_dir/'metal'/'z1-lm10-individual-Mg.hdf5'
h5.File.dump_to(fout,
    {
        'r_p': r_ps,
        'ang': angs,
        'col_dens': col_denss,
    })

In [ ]:
def err_prop(u, v, du, dv):
    out = u/v
    err = np.abs(u*dv - v*du) / v**2
    return out, err

In [ ]:
fig, axs = plot.subplots((2,1), share=(True, False), space=(0.25, .03), 
                         figsize=(6.5, 6.5),
                         margin=[0.02, 0.025, 0.085, 0.125], 
                         layout='none', ratios=(None, [1., .5]))
axs_f = axs.flat
cs = cs_named['b', 'r']
kws = dict(lw=3), dict(lw=1.5, ls=(0, (2,1.25)))
labs_file = [r'$z=0$', r'$z=1$']
labs_sel = [r'$|\theta_{\rm p}|<45^\circ$', r'$|\theta_{\rm p}| \geqslant 45^\circ$']

for i_file, file_name in enumerate(['z0-lm10-individual-Mg.hdf5', 
                               'z1-lm10-individual-Mg.hdf5'
                               ]):
    
    data = h5.File.load_from(sim_dir/'metal'/file_name)
    r_ps, angs, col_denss = data['r_p'], data['ang'], data['col_dens']

    ax = axs_f[0]
    sel_orth = np.abs(angs-np.pi/2) < np.pi/4.
    outs_sum = []
    for i_sel, sel in enumerate([sel_orth, ~sel_orth]):    
        xs = Num.safe_lg(r_ps[sel])
        ys = col_denss[sel]
        xs_p = np.linspace(1., 3., 128)
        ys_p, es_lo, es_hi = stats.KernelRegression1D.by_local_kernel(
            xs, ys, xs_p, max_dx=.4, reduce='errorbar', 
            kernel=.2,)['y'][0].T

        ax.c(cs[i_sel])
        if i_sel == 0:
            # mask = ys_p > 1.0e13
            mask = ys_p > 1.0e11
        xs_p, ys_p, es_lo, es_hi = xs_p[mask], ys_p[mask], es_lo[mask], es_hi[mask]
        xs_p = 10.0**xs_p
        
        lab = labs_file[i_file] + r'$,\,$' + labs_sel[i_sel]
        if i_file == 0:
            ax.plot(xs_p, ys_p, **kws[i_file], label=lab)
            ax.fill_between(xs_p, ys_p-es_lo, ys_p+es_hi, 
                            alpha=0.2, lw=0.2)
        else:
            ax.plot(xs_p, ys_p, **kws[i_file], label=lab)

        outs_sum.append((xs_p, ys_p, es_lo, es_hi))

    x_disk, y_disk, el_disk, eh_disk = outs_sum[0]
    x_polar, y_polar, el_polar, eh_polar = outs_sum[1]
    y_ratio = y_polar / y_disk

    ax = axs_f[1]
    ax.c(cs_named['k'])
    lab = labs_file[i_file]
    if i_file == 0:
        ax.plot(x_disk, y_ratio, **kws[i_file], label=lab)
    else:
        ax.plot(x_disk, y_ratio, **kws[i_file], label=lab)

ax = axs_f[1]
x, y, el, eh = lan_2018_ratio['x', 'r', 'e_lo', 'e_hi']
ax.c('grey').fmt_marker('s', s=8, c=cs_named['grey'], elw=2.25).errorbar(x, y, 
    yerr=[el, eh], lw=0, capsize=0)
    # label=r'${\rm Lan\,2018}\,(z \sim 1)$'
      
leg_kw = dict(loc='ur', labelcolor='linecolor', 
              handlelength=1.25, labelspacing=0.,fontsize=13.5)
ax = axs_f[0]
ax.scale('log', 'log')\
    .lim([10., 1000.0], [10**12.75, 10**18.5])\
    .leg(**leg_kw)\
    .label(y=r'N_{\rm Mg}\, [{\rm cm}^{-2}]')
    #.label(y=r'N_{\rm O}\, [{\rm cm}^{-2}]')
    

ax = axs_f[1]
ax.lim(y=[-.25, 3.55])\
    .leg(**leg_kw, numpoints=1)\
    .label(r'r_{\rm p}\,[{\rm kpc}]', r'\rm Ratio')
    
axs.label_outer()
# plot.savefig(out_fig_dir/'metal-proj.pdf')

In [ ]:
x1 = Num.safe_lg(r_es+r_es[1]*.9)
x2 = theta_es
y = Num.safe_lg(np.median(n2, axis=(0,3)) / u_kpc2cm_3)

fig, ax = plot.subplots(1, figsize=4.5, margin=[0.1, 0.1, 0.1, 0.1], layout='none')

ax._raw.pcolormesh(x1, x2, y.T, vmin=-9., vmax=-3.)

In [ ]:
r_p_sizes = np.logspace(0., 3., 64)
axes_list = [[1.,0.,0.], [0.,1.,0.], [0.,0.,1.]],\
            [[1.,0.,0.], [0.,0.,-1.], [0.,1.,0.]]

outs = []

for i_r, r_p_size in enumerate(r_p_sizes):
    out = [] 
    for i_axes, axes in enumerate(axes_list):
        lf = TiltedDiskLocalFrame(r_p=[0., -r_p_size, 0.], axes=axes)
        integ = SightlineIntegrator(lf, n_med, r_es, theta_es, phi_es)
        col_dens = integ.integrate(z_max=1500.0, eps=.2) / u_kpc2cm_2
        out.append(col_dens)
    outs.append(out)

In [ ]:
fig, ax = plot.subplots(1, figsize=4.5, margin=[0.1, 0.1, 0.1, 0.1], layout='none')

N_1, N_2 = np.array(outs).T
ax.plot(r_p_sizes, N_1, c=cs_named['b'])
ax.plot(r_p_sizes, N_2, c=cs_named['r'])

ax.scale('log', 'log').lim([10., 1000.])

In [ ]:
## Check sightline random sampling
rng = Rng(10086)
angs = []
r_p_norms = []
for i in range(10000):
    lf = TiltedDiskLocalFrame.new_random(rng, 1500.0)
    ang = lf.angle_r_p2ez_p()
    angs.append(ang)
    
    r_p_norms.append(np.linalg.norm(lf.r_p))
    
fig, ax = plot.subplots(1, figsize=4.5, margin=[0.1, 0.1, 0.1, 0.1], layout='none')

#ax.hist(angs, bins=32, density=True)
ax.hist(r_p_norms, bins=32, density=True)

# Fitting streams

In [ ]:
%autoreload
from baryon_cycle.model import (SpinFit, VinFit,
    FaceOnRecon, EdgeOnRecon, fn_S, fn_P, fn_double_S, fn_double_PS,
    FittedVinProfile, FittedVoutProfile, VoutProfileFit,
    VoutMapFit, FittedVoutMap, VthetaMapFit, FittedVthetaMap)

In [ ]:
# all_data = h5.File.load_from(sim_dir / 'radial_profiles.hdf5')
# field2D_data = h5.File.load_from(sim_dir / '2Dmaps.hdf5')
# field3D_data = h5.File.load_from(sim_dir / '3Dmaps.hdf5')
# sim_info = sims.predefined[field3D_data['header/sim_name'].decode()]

In [ ]:
fields_1D = h5.File.load_from(sim_dir / '1Dprofiles_interp.hdf5')
fields_2D = h5.File.load_from(sim_dir / '2Dmaps_interp.hdf5')
fields_3D = h5.File.load_from(sim_dir / '3Dmaps_interp.hdf5')
sim_info = sims.predefined[fields_3D['header/sim_name'].decode()]

In [ ]:
z_key = 'z0'
lm_key = 'lm10'
ecosys = ecosys_infos[f'{z_key}/{lm_key}']
R_h, V_h, H = ecosys.R_h, ecosys.V_h, ecosys.H

## Face-on

In [ ]:
rs = fields_1D['header/r_edges']
r_cs = 0.5 * (rs[1:] + rs[:-1])
sel = r_cs < 1.5 * R_h

pf_data = fields_1D[f'{z_key}/{lm_key}']

H0 = sim_info.cosmology.hubble * 100.0 / 1.0e3     # km/s/kpc
v_H = r_cs * H0

v_rs = pf_data['v_r/disk/mean'] #- v_H
ys1 = v_rs/V_h

v_rs = pf_data['v_r/disk/median'] # - v_H
ys2 = v_rs/V_h

ys3 = v_H / V_h

In [ ]:
fig, ax = plot.subplots(1, figsize=4.5, margin=[0.1, 0.1, 0.1, 0.1], layout='none')

xs = r_cs / R_h
ax.plot(xs, ys1, c='k', lw=4, ls='--')
ax.plot(xs, ys2, c='r')

ax.plot(xs, ys3, c='k', lw=1)

ax.lim([0., 7.5])

In [ ]:
def _find_face_on_fit():
    rs = fields_1D['header/r_edges']
    r_cs = 0.5 * (rs[1:] + rs[:-1])
    sel = r_cs < 1.5 * R_h

    pf_data = fields_1D[f'{z_key}/{lm_key}']
    spins = pf_data['lambda/disk/median']
    xs, ys = r_cs[sel]/R_h, spins[sel]
    spin_fit = SpinFit(xs, ys)
    spin_fit.find_optim()
    #opt_pf = spin_fit.opt_pf
    #y_preds = sf.init_pf.y_at(xs)
    #y_preds2 = sf.opt_pf.y_at(xs)
    
    H0 = sim_info.cosmology.hubble * 100.0 / 1.0e3     # km/s/kpc
    v_H = r_cs * H0
    v_rs = pf_data['v_r/disk/median'] - v_H
    ys = - v_rs[sel]/V_h
    vin_fit = VinFit(xs, ys)
    vin_fit.find_optim()
    
    return spin_fit, vin_fit

In [ ]:
spin_fit, vin_fit = _find_face_on_fit()

In [ ]:
spin_fit.opt_pf

In [ ]:
vin_fit.opt_pf

In [ ]:
ys_pred = vin_fit.init_pf.y_at(xs)
ys_pred2 = vin_fit.opt_pf.y_at(xs)

In [ ]:
fig, ax = plot.subplots(1, figsize=4.5, margin=[0.1, 0.1, 0.1, 0.1], layout='none')

ax.plot(xs, ys)
ax.plot(xs, ys_pred)
ax.plot(xs, ys_pred2)

#ax.lim([0., 0.1])

In [ ]:
sim_info = sims.predefined['tng_50_1']
recon = FaceOnRecon(field3D_data['header']['bin_edges/CGM']['x', 'y'], 
                    spin_fit.opt_pf, vin_fit.opt_pf, ecosys_defs['z0/lm10'])
recon.run()

In [ ]:
recon = FaceOnRecon(fields_3D['header']['bin_edges/IGM']['x', 'y'], 
                    spin_fit.opt_pf, vin_fit.opt_pf, ecosys)
recon.run()

In [ ]:
fig, axs = plot.subplots((1,3), share=False, space=0.02, subsize=(4.5, 4.5), 
                         margin=[0.02, 0.02, 0.125, 0.1], layout='none')
axs_f = axs.flat

norm=mcolors.Normalize(vmin=0,vmax=120)

xs, ys = recon.x_cs, recon.y_cs
v_x, v_y = fields_3D['face_on/IGM']['v_x/median', 'v_y/median']
v_x, v_y = v_x.T, v_y.T
v_x1, v_y1 = recon.data['v_xs', 'v_ys']
dv_x, dv_y = v_x - v_x1, v_y - v_y1

stream_kw = dict(cmap='rainbow',norm=norm,linewidth=.75,
                 density=8,
                 #density=4,
                 arrowsize=.8)
ax = axs_f[0]
v_norm = np.hypot(v_x1, v_y1)
strm= ax._raw.streamplot(xs, ys, v_x1, v_y1, color=v_norm, **stream_kw)

ax = axs_f[1]
v_norm = np.hypot(v_x, v_y)
strm= ax._raw.streamplot(xs, ys, v_x, v_y, color=v_norm, **stream_kw)

ax = axs_f[2]
v_norm = np.hypot(dv_x, dv_y)
dv = np.hypot(dv_x, dv_y)
ax._raw.pcolormesh(xs, ys, dv, cmap='rainbow', norm=norm, rasterized=True)

for ax in axs_f:
    ax.lim([-675, 675], [-675, 675]).label(r'x\,[{\rm kpc}]', r'y\,[{\rm kpc}]')
    ax._raw.set_aspect('equal')
    ax._raw.grid(False)
    ax._raw.set_facecolor('k')
    ax._raw.tick_params(axis='both', color='w', width=1, length=5)
    axs.label_outer()
    
plot.savefig(out_fig_dir/'model-face-on.pdf')

In [ ]:
rs = all_data['header/r_edges']
r_cs = 0.5 * (rs[1:] + rs[:-1])


fig, axs = plot.subplots((2,1), share=False, space=0.25, 
                         subsize=(10.5, 5.0), margin=[0.1, 0.1, 0.1, 0.1], layout='none')
axs_f = axs.flat

m_cs = 'b', 'g', 'r'
for i_m, lm_key in enumerate(['lm9', 'lm10', 'lm11']):
    ax = axs_f[0]
    R_h, V_h = ecosys_defs[f'z0/{lm_key}']['R_h', 'V_h']    
    ys = all_data[f'z0/{lm_key}/lambda/major']
    xs = r_cs / R_h
    ax.plot(xs, ys, lw=i_m+1, c=m_cs[i_m])
    
    ax = axs_f[1]
    ys = all_data[f'z0/{lm_key}/v_r/major'] / V_h
    ax.plot(xs, ys, lw=i_m+1, c=m_cs[i_m])

ax = axs_f[0]
ax.lim([.01, 2.0], [0.001, 0.2]).label(r'r/R_{\rm h}', r'\lambda')

ax = axs_f[1]
ax.lim([.01, 2.0], [-0.3, 0.225]).label(r'r/R_{\rm h}', r'v_r/V_{\rm h}')

In [ ]:
np.array(1.).item()

In [ ]:
0.075 * 1.35

In [ ]:
rs = all_data['header/r_edges']
r_cs = 0.5 * (rs[1:] + rs[:-1])

fig, axs = plot.subplots((2,2), share=False, space=0.25, subsize=(5.0, 5.0), margin=[0.1, 0.1, 0.1, 0.1], layout='none')
axs_f = axs.flat

m_cs = 'b', 'g', 'r'
H0 = sim_info.cosmology.hubble * 100.0 / 1.0e3     # km/s/kpc
for i_m, lm_key in enumerate(['lm9', 'lm10', 'lm11']):
    ax = axs_f[0]
    val = all_data[f'z0/{lm_key}/v_r/major']
    R_h, V_h = ecosys_defs[f'z0/{lm_key}']['R_h', 'V_h']
    v_H = r_cs * H0
    val = val - v_H
    xs = r_cs / R_h
    ys = - val / V_h
    
    ax.plot(xs, ys, lw=i_m+1, c=m_cs[i_m])

    ys1 = 0.4*(xs/0.5)**-0.5
    ax.plot(xs, ys1, c='k')
    
    #dxs = (.645 - xs).clip(0.)/.4
    #s = np.exp(-dxs**1.5) 
    s = 1. - ((0.69-xs).clip(0.)/.505)**1.8
    s = s.clip(0.)
    #s = np.exp(- ((0.69-xs).clip(0.)/.305)**3.)
    ys2 = ys1 * s
    #ax.plot(xs, ys2, c='k', ls='--')
    
    #dy3 = 0.06*((0.15-xs)/0.15).clip(0.)**2.25
    dy3 = 0.075 * np.exp(-xs/0.03) * (xs/0.03)**.25 * 1.35
    ys3 = ys2 + dy3
    ax.plot(xs, ys3, c='k', ls=':')
    
    if i_m == 1:
        ax = axs_f[1]
        yr = ys / ys1
        ax.plot(xs, yr, c=m_cs[i_m])
        ax.plot(xs, s, c='k', ls='-')
    
        ax = axs_f[2]
        dy = ys - ys2
        ax.plot(xs, dy, c=m_cs[i_m])
        
        dy = ys - ys3
        ax.plot(xs, dy, c='k')
    
    
ax = axs_f[0]
ax.lim([.001, 12.5], [-.1, .75])#.scale('log')
#ax.lim([0.1, 2.], [-.05,.5])#.scale('log')

ax = axs_f[1]
ax.lim([.001, 1.], [0., 1.2])

ax = axs_f[2]
ax.lim([.001, 2.5*.1], [-.1, 0.1])

In [ ]:
rs = all_data['header/r_edges']
r_cs = 0.5 * (rs[1:] + rs[:-1])
lm_key = 'lm10'
vals = all_data[f'z0/{lm_key}/lambda/major']
R_h = ecosys_defs[f'z0/{lm_key}/R_h']

sel = r_cs < 1.5 * R_h
xs, ys = r_cs[sel]/R_h, vals[sel]

sf = SpinFit(xs, ys)
y_preds = sf.init_pf.y_at(xs)
sf.find_optim()
y_preds2 = sf.opt_pf.y_at(xs)
opt_pf = sf.opt_pf

In [ ]:
fig, ax = plot.subplots(1, figsize=(10.6, 4.5), margin=[0.1, 0.1, 0.1, 0.1], layout='none')

ax.plot(xs, ys, c='C0', lw=2, label='data')
ax.plot(xs, y_preds, c='C1', lw=2, ls='--', label='fit')
ax.plot(xs, y_preds2, lw=3)
#ax.lim([])

## Edge-on

In [ ]:
class VoutMapFit2:
    def __init__(self, thetas: np.ndarray, xs: np.ndarray,
                 y_map: np.ndarray,
                 vin_pf: FittedVinProfile,
                 vout_pf: FittedVoutProfile,
                 wgt_map: np.ndarray|None = None):
        '''
        Here x is normalized by R_h.
        Theta [rad] is inclination angle.
        y_map is normalized by V_h, does not include Hubble flow.
        '''
        theta_map, x_map = np.meshgrid(thetas, xs, indexing='ij')
        if wgt_map is None:
            wgt_map = np.ones_like(y_map)
        else:
            wgt_map = np.array(wgt_map)
        
        self.thetas = np.array(thetas)
        self.xs = np.array(xs)
        self.theta_map = theta_map
        self.x_map = x_map
        self.y_map = np.array(y_map)
        self.wgt_map = wgt_map
        self.vin_pf = vin_pf
        self.vout_pf = vout_pf
        
        self.init_map = FittedVoutMap(
            vin_pf=vin_pf,
            vout_pf=vout_pf,
            a_in=1.,
            s_in=0.5,
            a_out=1.,
            s_out=0.5,
        )
        
    def find_optim(self):
        param0 = self.init_map.as_tuple() + self.vout_pf.as_tuple()
        bounds = [
            (-0.99, 15.0),
            (0.01, np.pi),
            (-0.99, 15.0),
            (0.01, np.pi),
            
            (0.0, 10.0),
            (0.001, 10.0),
            (0.001, 5.0),
            (-0.99, 5.0),
            (0.001, 5.0),
            (-0.99, 5.0),
            (-3.0, 3.0),
        ]
        out = minimize(self.f_obj, param0,
                       method='Nelder-Mead', bounds=bounds,
            options={'maxiter':50000})
        if not out.success:
            print(out.message)
        vout_pf = FittedVoutProfile(*out.x[4:])
        self.opt_map = FittedVoutMap(self.vin_pf, vout_pf, *out.x[:4])
        
    def f_obj(self, params):
        wgt_map = self.wgt_map
        vout_pf = FittedVoutProfile(*params[4:])
        fit = FittedVoutMap(self.vin_pf, vout_pf, *params[:4])
        y_map_pred = fit.y_at(self.theta_map, self.x_map)
        res = self.y_map - y_map_pred
        
        prior = 0.
        
        return (res*res*wgt_map).sum() + prior

In [ ]:
class EdgeOnData:
    def __init__(self, maps: DataDict[str, np.ndarray], 
                 profs: DataDict[str, np.ndarray],
                 header: DataDict[str, np.ndarray], 
                 ecosys: EcosysInfo):
        
        self.maps = maps
        self.profs = profs
        self.header = header
        self.ecosys = ecosys
        
        self._set_up_2D()
        self._set_up_1D()
        
    def _set_up_2D(self):
        maps, header, ecosys = self.maps, self.header, self.ecosys
        R_h, V_h, H = ecosys.R_h, ecosys.V_h, ecosys.H
        
        v_r_map =  maps['v_r/median'].T
        v_theta_map = maps['v_theta/median'].T
        theta_es, r_es = header['theta_edges', 'r_edges']
        inc_es = -theta_es + np.pi/2
        dincs = inc_es[1:] - inc_es[:-1]
        drs = r_es[1:] - r_es[:-1]
        
        r_cs = .5 * (r_es[:-1] + r_es[1:])
        inc_cs = .5 * (inc_es[:-1] + inc_es[1:])
        inc_cs0 = inc_cs.copy()
        inc_cs[0], inc_cs[-1] = -np.pi/2., np.pi/2.
        
        v_Hubble = H * r_cs
        v_out_map = v_r_map - v_Hubble
        
        y_r_map = v_r_map / V_h
        y_out_map = v_out_map / V_h
        y_theta_map = v_theta_map / V_h
        
        xs = r_cs / R_h
        wgt_map = dincs[:, None] * drs[None, :]
        wgt_map /= wgt_map.mean()
        
        self.v_Hubble = v_Hubble
        self.inc_cs0 = inc_cs0
        self.inc_cs = inc_cs
        self.dincs = dincs
        self.r_cs = r_cs
        self.drs = drs
        self.xs = xs
        self.v_r_map = v_r_map
        self.v_out_map = v_out_map
        self.v_theta_map = v_theta_map
        self.y_r_map = y_r_map
        self.y_out_map = y_out_map
        self.y_theta_map = y_theta_map
        self.wgt_map = wgt_map
        
    def _set_up_1D(self):
        profs, ecosys = self.profs, self.ecosys
        v_Hubble, V_h = self.v_Hubble, ecosys.V_h
        
        v_r_polar = profs['v_r/polar/median']
        v_out_polar = v_r_polar - v_Hubble
        y_r_polar = v_r_polar / V_h
        y_out_polar = v_out_polar / V_h
        
        inc_cs = self.inc_cs
        n_incs = len(inc_cs)
        inc_cs_1D = inc_cs[n_incs//2:].copy()
        inc_cs_1D[0] = 0.
        
        self.v_r_polar = v_r_polar
        self.v_out_polar = v_out_polar
        self.y_r_polar = y_r_polar
        self.y_out_polar = y_out_polar
        self.inc_cs_1D = inc_cs_1D
        
    def at_r(self, x: float):
        arg = np.abs(self.xs-x).argmin()
        return DataDict({
            'v_r': self.v_r_map[:, arg],
            'v_out': self.v_out_map[:, arg],
            'v_theta': self.v_theta_map[:, arg],
            'y_r': self.y_r_map[:, arg],
            'y_out': self.y_out_map[:, arg],
            'y_theta': self.y_theta_map[:, arg],
        })
        
    def at_inc(self, inc: float):
        arg = np.abs(self.inc_cs-inc).argmin()
        return DataDict({
            'v_r': self.v_r_map[arg],
            'v_out': self.v_out_map[arg],
            'v_theta': self.v_theta_map[arg],
            'y_r': self.y_r_map[arg],
            'y_out': self.y_out_map[arg],
            'y_theta': self.y_theta_map[arg],
        })

maps = fields_2D[f'{z_key}/{lm_key}']
profs = fields_1D[f'{z_key}/{lm_key}']
header = fields_2D['header']
ecosys = ecosys_infos[f'{z_key}/{lm_key}']

In [ ]:
d_edge = EdgeOnData(maps, profs, header, ecosys)

In [ ]:
xs, ys = d_edge.xs, d_edge.y_out_polar
sel = xs < 3.
xs_sel, ys_sel = xs[sel], ys[sel]
vout_fit = VoutProfileFit(xs_sel, ys_sel)
vout_fit.find_optim()
vout_fit.opt_pf

In [ ]:
xs, ys = d_edge.xs, d_edge.y_out_polar

pf = FittedVoutProfile(1.6, 0.03, 0.02, .5, .16, 0.1, .275)
ys_pred = pf.y_at(xs)

pf2 = vout_fit.opt_pf
ys_pred2 = pf2.y_at(xs)

fig, axs = plot.subplots((4,1), share=False, space=0.25, subsize=(8.5, 3.20), margin=[0.1, 0.1, 0.1, 0.1], layout='none')
axs_f = axs.flat

x_uplims = .2, 1., 4., 15.
for i_x, x_uplim in enumerate(x_uplims):
    ax = axs_f[i_x]
    ax.plot(xs, ys)
    ax.plot(xs, ys_pred, c='r')
    ax.plot(xs, ys_pred2, c='b')

    ax.lim([0.00, x_uplim])

In [ ]:
xs, ys = d_edge.xs, d_edge.y_out_polar
sel = xs < 3.
xs_sel, ys_sel = xs[sel], ys[sel]

incs = d_edge.inc_cs
y_map = d_edge.y_out_map
y_map_sel = y_map[:, sel]
wgts = np.zeros_like(y_map_sel) + d_edge.dincs[:, None]  #d_edge.wgt_map[:, sel]

voutmap_fit = VoutMapFit(
    incs, xs_sel, y_map_sel, vin_fit.opt_pf, vout_fit.opt_pf, wgts)
voutmap_fit.find_optim()
voutmap_fit.opt_map

In [ ]:
voutmap_fit = VoutMapFit2(
    incs, xs_sel, y_map_sel, vin_fit.opt_pf, vout_fit.opt_pf, wgts)
voutmap_fit.find_optim()
voutmap_fit.opt_map

In [ ]:
xs_show = 0.05, .1, .25, .5, 1., 2.
cs = mpl.cm.rainbow(np.linspace(0.2,1,len(xs_show)))

fig, axs = plot.subplots((1,3), share=False, space=0.25, subsize=(5.0, 5.0), margin=[0.1, 0.1, 0.1, 0.1], layout='none')
axs_f = axs.flat

for i_x, x in enumerate(xs_show):
    y = d_edge.at_r(x)['y_out']
    yp1 = voutmap_fit.init_map.y_at(incs, np.zeros_like(incs)+x)
    yp2 = voutmap_fit.opt_map.y_at(incs, np.zeros_like(incs)+x)
    yps = yp1, yp2
    for i_ax, ax in enumerate(axs_f[:2]):
        ax.plot(incs, y, c=cs[i_x])
        ax.plot(incs, yps[i_ax], c=cs[i_x], ls='--')
    
axs.lim([-np.pi/2., np.pi/2.])

In [ ]:
y_map_ini = voutmap_fit.init_map.y_at(voutmap_fit.theta_map, voutmap_fit.x_map)
y_map_opt = voutmap_fit.opt_map.y_at(voutmap_fit.theta_map, voutmap_fit.x_map)

fig, axs = plot.subplots((2,2), share=False, space=0.25, subsize=(5.0, 5.0), margin=[0.1, 0.1, 0.1, 0.1], layout='none')
axs_f = axs.flat

for i_y, y in enumerate([y_map_sel, y_map_ini, y_map_opt, y_map_sel-y_map_opt]):
    ax = axs_f[i_y]
    ax._raw.pcolormesh(incs, xs_sel, y.T, cmap='rainbow', 
                       rasterized=True, vmin=-.2, vmax=1.)

#axs.lim(y=[0., 600.])

In [ ]:
xs, incs = d_edge.xs, d_edge.inc_cs0
sel = xs < 3.5

y_map = d_edge.y_theta_map
y_map_sel = y_map[:, sel]
xs_sel = xs[sel]
wgts = np.zeros_like(y_map_sel) + d_edge.dincs[:, None]  #d_edge.wgt_map[:, sel]
#wgts = None

vthetamap_fit = VthetaMapFit(incs, xs_sel, y_map_sel, wgts)
vthetamap_fit.find_optim()
vthetamap_fit.opt_map

In [ ]:
from dataclasses import asdict
for k, v in asdict(vthetamap_fit.opt_map).items():
    print(f'{k}={v:.3f}')

In [ ]:
xs = d_edge.xs
xs_show = 0.01, .05, .1, .15, .25, .5, .75, 1., 1.5, 2.
cs = mpl.cm.rainbow(np.linspace(0.2,1,len(xs_show)))

fig, axs = plot.subplots((1,3), share=False, space=0.25, subsize=(5.0, 5.0), margin=[0.1, 0.1, 0.1, 0.1], layout='none')
axs_f = axs.flat

for i_x, x in enumerate(xs_show):
    y = d_edge.at_r(x)['y_theta']
    yp1 = vthetamap_fit.init_map.y_at(incs, np.zeros_like(incs)+x)
    yp2 = vthetamap_fit.opt_map.y_at(incs, np.zeros_like(incs)+x)
    yps = yp1, yp2
    for i_ax, ax in enumerate(axs_f[:2]):
        ax.plot(incs, y, c=cs[i_x])
        ax.plot(incs, yps[i_ax], c=cs[i_x], ls='--')
    
axs.lim([-np.pi/2., np.pi/2.])

In [ ]:
incs_show = .1, .25, .5, 1., 1.5
cs = mpl.cm.rainbow(np.linspace(0.2,1,len(incs_show)))

fig, axs = plot.subplots((1,2), share=False, space=0.25, subsize=(7.5, 5.0), margin=[0.1, 0.1, 0.1, 0.1], layout='none')
axs_f = axs.flat

for i_inc, inc in enumerate(incs_show):
    y = d_edge.at_inc(inc)['y_theta']
    yp1 = vthetamap_fit.init_map.y_at(np.zeros_like(xs)+inc, xs)
    yp2 = vthetamap_fit.opt_map.y_at(np.zeros_like(xs)+inc, xs)
    yps = yp1, yp2   
    for i_ax, ax in enumerate(axs_f[:2]):
        ax.plot(xs, y, c=cs[i_inc])
        ax.plot(xs, yps[i_ax], c=cs[i_inc], ls='--')
    
axs.lim([0., .25])

fig, axs = plot.subplots((1,2), share=False, space=0.25, subsize=(7.5, 5.0), margin=[0.1, 0.1, 0.1, 0.1], layout='none')
axs_f = axs.flat

for i_inc, inc in enumerate(incs_show):
    y = d_edge.at_inc(inc)['y_theta']
    yp1 = vthetamap_fit.init_map.y_at(np.zeros_like(xs)+inc, xs)
    yp2 = vthetamap_fit.opt_map.y_at(np.zeros_like(xs)+inc, xs)
    yps = yp1, yp2   
    for i_ax, ax in enumerate(axs_f[:2]):
        ax.plot(xs, y, c=cs[i_inc])
        ax.plot(xs, yps[i_ax], c=cs[i_inc], ls='--')
    
axs.lim([0., 3.5], [-0.1, 0.1])

In [ ]:
y_map_ini = vthetamap_fit.init_map.y_at(vthetamap_fit.theta_map, vthetamap_fit.x_map)
y_map_opt = vthetamap_fit.opt_map.y_at(vthetamap_fit.theta_map, vthetamap_fit.x_map)

fig, axs = plot.subplots((2,2), share=False, space=0.25, subsize=(5.0, 5.0), margin=[0.1, 0.1, 0.1, 0.1], layout='none')
axs_f = axs.flat

for i_y, y in enumerate([y_map_sel, y_map_ini, y_map_opt, y_map_opt-y_map_sel]):
    ax = axs_f[i_y]
    ax._raw.pcolormesh(incs, xs_sel, y.T, cmap='RdBu_r', 
                       rasterized=True, vmin=-.3, vmax=.3)

axs.lim(y=[0., 1.5])

In [ ]:
recon = EdgeOnRecon(fields_3D['header']['bin_edges/IGM']['x', 'y'], 
                    voutmap_fit.opt_map, vthetamap_fit.opt_map,
                    ecosys)
recon.run()

In [ ]:
fig, axs = plot.subplots((1,3), share=False, space=0.02, subsize=(4.5, 4.5), 
                         margin=[0.02, 0.02, 0.125, 0.1], layout='none')
axs_f = axs.flat

norm=mcolors.Normalize(vmin=0,vmax=60)

xs, ys = recon.x_cs, recon.y_cs
v_x, v_y = fields_3D['edge_on/IGM']['v_x/median', 'v_y/median']
v_x, v_y = v_x.T, v_y.T
v_x1, v_y1 = recon.data['v_xs', 'v_ys']
dv_x, dv_y = v_x1 - v_x, v_y1 - v_y

stream_kw = dict(cmap='rainbow',norm=norm,linewidth=1.,
                 #density=8,
                 density=4,
                 arrowsize=.8)
ax = axs_f[0]
v_norm = np.hypot(v_x1, v_y1)
strm= ax._raw.streamplot(xs, ys, v_x1, v_y1, color=v_norm, **stream_kw)

ax = axs_f[1]
v_norm = np.hypot(v_x, v_y)
strm= ax._raw.streamplot(xs, ys, v_x, v_y, color=v_norm, **stream_kw)

ax = axs_f[2]
v_norm = np.hypot(dv_x, dv_y)
dv = np.hypot(dv_x, dv_y)
ax._raw.pcolormesh(xs, ys, dv, cmap='rainbow', norm=norm, rasterized=True)

for ax in axs_f:
    ax.lim([-675, 675], [-675, 675]).label(r'x\,[{\rm kpc}]', r'z\,[{\rm kpc}]')
    
    ax._raw.set_aspect('equal')
    ax._raw.grid(False)
    ax._raw.set_facecolor('k')
    ax._raw.tick_params(axis='both', color='w', width=1, length=5)
    axs.label_outer()
    
# plot.savefig(out_fig_dir/'model-edge-on.pdf')

In [ ]:
recon = EdgeOnRecon(fields_3D['header']['bin_edges/CGM']['x', 'y'], 
                    voutmap_fit.opt_map, vthetamap_fit.opt_map,
                    ecosys)
recon.run()

In [ ]:
fig, axs = plot.subplots((1,3), share=False, space=0.02, subsize=(4.5, 4.5), 
                         margin=[0.02, 0.02, 0.125, 0.1], layout='none')
axs_f = axs.flat

norm=mcolors.Normalize(vmin=0,vmax=60)

xs, ys = recon.x_cs, recon.y_cs
v_x, v_y = fields_3D['edge_on/CGM']['v_x/median', 'v_y/median']
v_x, v_y = v_x.T, v_y.T
v_x1, v_y1 = recon.data['v_xs', 'v_ys']
dv_x, dv_y = v_x1 - v_x, v_y1 - v_y

stream_kw = dict(cmap='rainbow',norm=norm,linewidth=1.,
                 #density=8,
                 density=4,
                 arrowsize=.8)
ax = axs_f[0]
v_norm = np.hypot(v_x1, v_y1)
strm= ax._raw.streamplot(xs, ys, v_x1, v_y1, color=v_norm, **stream_kw)

ax = axs_f[1]
v_norm = np.hypot(v_x, v_y)
strm= ax._raw.streamplot(xs, ys, v_x, v_y, color=v_norm, **stream_kw)

ax = axs_f[2]
v_norm = np.hypot(dv_x, dv_y)
#dv = np.hypot(dv_x, dv_y)
dv = dv_y
norm = mcolors.Normalize(vmin=-60,vmax=60)
ax._raw.pcolormesh(xs, ys, dv, cmap='rainbow', norm=norm, rasterized=True)

for ax in axs_f:
    #ax.lim([-675, 675], [-675, 675]).label(r'x\,[{\rm kpc}]', r'z\,[{\rm kpc}]')
    
    ax._raw.set_aspect('equal')
    ax._raw.grid(False)
    ax._raw.set_facecolor('k')
    ax._raw.tick_params(axis='both', color='w', width=1, length=5)
    axs.label_outer()

### Outflow map

In [ ]:
fig, ax = plot.subplots(1, figsize=(14.5, 14.5), margin=[0.1, 0.1, 0.1, 0.1], layout='none')

cs = 'b', 'g', 'r'
for i_m, lm_key in enumerate(['lm9', 'lm10', 'lm11']):
    
    samp_key = f'z0/{lm_key}'

    prop_maps = field2D_data[samp_key]
    header = field2D_data['header']
    ecosys_def = ecosys_defs[samp_key]
    R_h, V_h, H = ecosys_def['R_h', 'V_h', 'H']

    v_r =  prop_maps['v_r'] 
    ys = v_r / V_h

    theta_edges, r_edges = header['theta_edges', 'r_edges']
    inc_edges = -theta_edges + np.pi/2
    s_inc_edges = np.sin(inc_edges)
    s_inc_cs = .5 * (s_inc_edges[:-1] + s_inc_edges[1:])
    r_cs = .5 * (r_edges[:-1] + r_edges[1:])
    xs = r_cs / R_h

    y_minor = all_data[samp_key]['v_r/minor'] / V_h
    r_es_1d = all_data['header']['r_edges']
    r_cs_1d = .5 * (r_es_1d[1:] + r_es_1d[:-1])
    xs_1d = r_cs_1d / R_h
    
    y_H = H * r_cs_1d * 1. / V_h
    dy = y_minor - y_H
    
    if i_m == 0:
        xsrel = xs
        yp1 = ((xsrel/0.04)**2. * .625).clip(0.)
        sel = xs > .04
        yp1[sel] =  np.exp(- (xsrel[sel] - 0.04)**.5/2.15) * .625
        ax.plot(xs, yp1)
    elif i_m == 1:
        fi
    yp1 = xs
    #y_cone = (v_r[:2].mean(0) + v_r[-2:].mean(0))*.5 / V_h
    #ax.plot(xs, y_cone, c=cs[i_m], lw=2)
    
    ax.plot(xs_1d, y_minor, c=cs[i_m], lw=2)
    ax.plot(xs_1d, y_H, c=cs[i_m], lw=5, ls='--')
    ax.plot(xs_1d, dy, c=cs[i_m], lw=2.5, ls='--')
    
ax.lim([0.001, 5.5*1], [-0.2, 3.5])

In [ ]:
%autoreload
from baryon_cycle.model import SpinFit, VinFit, VoutFit, VthetaFit, FaceOnRecon, EdgeOnRecon, FittedVinProfile, FittedVoutMap

In [ ]:
prop_maps = fields_2D[f'{z_key}/{lm_key}']
header = fields_2D['header']
v_r_map =  prop_maps['v_r/median'].T
#ys = v_r / V_h

theta_edges, r_edges = header['theta_edges', 'r_edges']
inc_edges = -theta_edges + np.pi/2
s_inc_edges = np.sin(inc_edges)
s_inc_cs = .5 * (s_inc_edges[:-1] + s_inc_edges[1:])
r_cs = .5 * (r_edges[:-1] + r_edges[1:])
xs = r_cs / R_h
inc_cs = np.arcsin(s_inc_cs)
#inc_map, x_map = np.meshgrid(inc_cs, xs, indexing='ij')
v_Hubble = H * r_cs

y_map = (v_r_map - v_Hubble)/V_h

sel = xs < 3.
xs = xs[sel]
y_map = y_map[:, sel]

vout_fit = VoutFit(inc_cs, xs, y_map, vin_fit.opt_pf)
vout_fit.find_optim()

In [ ]:
fig, ax = plot.subplots(1, figsize=4.5, margin=[0.1, 0.1, 0.1, 0.1], layout='none')

xs_show = [.01, 0.1, .5, 1., 1.5]
args = [np.abs(xs - x_val).argmin() for x_val in xs_show]
cs = 'r', 'g', 'b', 'm', 'k'
for i, arg in enumerate(args):
    y = y_map[:, arg]
    ax.plot(inc_cs, y, c=cs[i], alpha=0.3)

In [ ]:
opt = FittedVoutMap(vin_pf=vin_fit.opt_pf,
            A = y_map[[0,-1]][:,xs<1.].max(),
            a_1 = 1.,
            x_1 = 0.04, # 0.05, 
            dx_1 = 0.02,
            a_2 = -0.5,
            s_2 = 1.,
            a_in=1.,
            s_in=0.5,
            a_out=1.,
            s_out=1.5,)

In [ ]:
y_map_pred = vout_fit.opt_map.y_at(vout_fit.theta_map, vout_fit.x_map) 
vout_fit.opt_map
#y_map_pred = opt.y_at(vout_fit.theta_map, vout_fit.x_map) 
#opt

In [ ]:
y1 = .5 * (y_map_pred[0] + y_map_pred[-1])
y2 = .5 * (y_map[0] + y_map[-1])
#y1 = .5 * (y_map_pred[24] + y_map_pred[25])
#y2 = .5 * (y_map[24] + y_map[25])
fig, ax = plot.subplots(1, figsize=(10.5, 3.75), margin=[0.1, 0.1, 0.1, 0.1], layout='none')

ax.plot(xs, y1,)
ax.plot(xs, y2, c='r')

ax.lim([0., .25])
# ax.scale()

In [ ]:
fig, ax = plot.subplots(1, figsize=(14.5, 14.5), margin=[0.1, 0.1, 0.1, 0.1], layout='none')

cs = 'b', 'g', 'r'
fig, ax = plot.subplots(1, figsize=4.5, margin=[0.1, 0.1, 0.1, 0.1], layout='none')


In [ ]:
recon = EdgeOnRecon(field3D_data['header']['bin_edges/CGM']['x', 'y'], 
                    vout_fit.opt_map, ecosys_def)
recon.run()

In [ ]:
fig, axs = plot.subplots((2,2), share=False, space=0.25, subsize=(5.0, 5.0), margin=[0.1, 0.1, 0.1, 0.1], layout='none')
axs_f = axs.flat

norm=mcolors.Normalize(vmin=0,vmax=120)

xs, ys = recon.x_cs, recon.y_cs
v_x, v_y = field3D_data['edge_on/CGM']['v_x', 'v_y']
v_x1, v_y1, rs, incs, R_h, e_thetas = recon.data['v_xs', 'v_ys', 'rs', 'incs', 'R_h', 'e_thetas']
dv_x, dv_y = v_x1 - v_x, v_y1 - v_y

x_rs = rs/R_h
v_theta = v_x * e_thetas[0] + v_y * e_thetas[1]
dv_x1, dv_y1 = v_theta * e_thetas[0], v_theta * e_thetas[1]

stream_kw = dict(
    cmap='rainbow',norm=norm,linewidth=1.,density=1,
                         arrowsize=.8)
ax = axs_f[0]
v_x2, v_y2 = v_x1 + dv_x1, v_y1 + dv_y1
v_norm = np.hypot(v_x2, v_y2)
strm= ax._raw.streamplot(xs, ys, v_x2, v_y2, color=v_norm, **stream_kw)

ax = axs_f[1]
v_norm = np.hypot(v_x, v_y)
strm= ax._raw.streamplot(xs, ys, v_x, v_y, color=v_norm, **stream_kw)

ax = axs_f[2]
v_norm = np.hypot(dv_x, dv_y)
strm= ax._raw.streamplot(xs, ys, dv_x, dv_y, color=v_norm, **stream_kw)

for ax in axs_f:
    ax.lim([-199, 199], [-199, 199])#label(r'x\,[{\rm kpc}]', r'y\,[{\rm kpc}]')\    
    ax._raw.set_aspect('equal')
    ax._raw.grid(False)
    ax._raw.set_facecolor('k')
    ax._raw.yaxis.tick_right()
    ax._raw.tick_params(axis='both', color='w', width=1, length=5)

In [ ]:
fig, ax = plot.subplots(1, figsize=4.5, margin=[0.1, 0.1, 0.1, 0.1], layout='none')

ax.hist(v_theta.ravel()/V_h, bins=50, range=(-0.25, 0.25), density=True)

In [ ]:
recon = EdgeOnRecon(field3D_data['header']['bin_edges/ISM']['x', 'y'], 
                    vout_fit.opt_map, ecosys_def)
recon.run()

xs, ys = recon.x_cs, recon.y_cs
v_x, v_y = field3D_data['edge_on/ISM']['v_x', 'v_y']
v_x1, v_y1, rs, incs, R_h, e_thetas = recon.data['v_xs', 'v_ys', 'rs', 'incs', 'R_h', 'e_thetas']

x_rs = rs/R_h
dv_x, dv_y = v_x1 - v_x, v_y1 - v_y
v_theta = v_x * e_thetas[0] + v_y * e_thetas[1]

fig, ax = plot.subplots(1, figsize=6.5, margin=[0.1, 0.1, 0.1, 0.1], layout='none')

lt = ax._raw.pcolormesh(xs, ys, v_theta/V_h, cmap='bwr', vmin=-.2, vmax=.2)

ax._raw.set_aspect('equal')
fig.colorbar(lt, ax=ax._raw, pad=.1)

In [ ]:
prop_maps = fields_2D[f'{z_key}/{lm_key}']
header = fields_2D['header']

v_theta =  prop_maps['v_theta/median'].T 
ys = v_theta / V_h

theta_edges, r_edges = header['theta_edges', 'r_edges']
inc_edges = -theta_edges + np.pi/2
s_inc_edges = np.sin(inc_edges)
s_inc_cs = .5 * (s_inc_edges[:-1] + s_inc_edges[1:])
inc_cs = np.arcsin(s_inc_cs)
r_cs = .5 * (r_edges[:-1] + r_edges[1:])
xs = r_cs / R_h

sel = xs < 4.
vtheta_fit = VthetaFit(inc_cs, xs, ys)
vtheta_fit.find_optim()
vtheta_fit.opt_map

In [ ]:
m = vtheta_fit.opt_map
m.x_c, m.s_c2, m.x_b, m.s_b2

In [ ]:
fig, axs = plot.subplots((2,2), share=False, space=0.25, subsize=(5.0, 5.0), margin=[0.1, 0.1, 0.1, 0.1], layout='none')
axs_f = axs.flat

y_map = ys 
y_ini = vtheta_fit.init_map.y_at(vtheta_fit.theta_map, vtheta_fit.x_map)
y_opt = vtheta_fit.opt_map.y_at(vtheta_fit.theta_map, vtheta_fit.x_map)

kw = dict(cmap='RdBu_r', vmin=-0.25, vmax=0.25)

ax = axs_f[0]
ax._raw.pcolormesh(inc_cs, xs, y_map.T, **kw)

ax = axs_f[1]
ax._raw.pcolormesh(inc_cs, xs, y_ini.T, **kw)

ax = axs_f[2]
ax._raw.pcolormesh(inc_cs, xs, y_opt.T, **kw)

ax = axs_f[3]
ax._raw.pcolormesh(inc_cs, xs, (y_opt-y_map).T, **kw)

axs.lim(y=[0., 5.5])

In [ ]:
fig, ax = plot.subplots(1, figsize=4.5, margin=[0.1, 0.1, 0.1, 0.1], layout='none')

y2 = y_map[12]
y3 = y_opt[12]

ax.plot(xs, -y2)
ax.plot(xs, -y3, c='r')

y_pos = vtheta_fit.opt_map.y_pos_at(xs)
y_neg = -vtheta_fit.opt_map.y_neg_at(xs)
ax.plot(xs, y_pos, c='g')
ax.plot(xs, y_neg, c='b')

ax.lim([0., .25])
# ax.scale()

In [ ]:
x_es, y_es = fields_3D['header']['bin_edges/CGM']['x', 'y']

In [ ]:
x_cs, y_cs = 0.5 * (x_es[1:] + x_es[:-1]), 0.5 * (y_es[1:] + y_es[:-1])

In [ ]:
v_x, v_y = fields_3D['face_on/CGM']['v_x/median', 'v_y/median']
v_x, v_y = v_x.T, v_y.T

In [ ]:
recon = EdgeOnRecon(fields_3D['header']['bin_edges/IGM']['x', 'y'], 
                    vout_fit.opt_map, vtheta_fit.opt_map,
                    ecosys)
recon.run()

In [ ]:
fig, axs = plot.subplots((1,3), share=False, space=0.02, subsize=(4.5, 4.5), 
                         margin=[0.02, 0.02, 0.125, 0.1], layout='none')
axs_f = axs.flat

norm=mcolors.Normalize(vmin=0,vmax=60)

xs, ys = recon.x_cs, recon.y_cs
v_x, v_y = fields_3D['edge_on/IGM']['v_x/median', 'v_y/median']
v_x, v_y = v_x.T, v_y.T
v_x1, v_y1 = recon.data['v_xs', 'v_ys']
dv_x, dv_y = v_x1 - v_x, v_y1 - v_y

stream_kw = dict(cmap='rainbow',norm=norm,linewidth=1.,
                 #density=5,
                 density=2,
                 arrowsize=.8)
ax = axs_f[0]
v_norm = np.hypot(v_x1, v_y1)
strm= ax._raw.streamplot(xs, ys, v_x1, v_y1, color=v_norm, **stream_kw)

ax = axs_f[1]
v_norm = np.hypot(v_x, v_y)
strm= ax._raw.streamplot(xs, ys, v_x, v_y, color=v_norm, **stream_kw)

ax = axs_f[2]
v_norm = np.hypot(dv_x, dv_y)
dv = np.hypot(dv_x, dv_y)
ax._raw.pcolormesh(xs, ys, dv, cmap='rainbow', norm=norm, rasterized=True)

for ax in axs_f:
    ax.lim([-675, 675], [-675, 675]).label(r'x\,[{\rm kpc}]', r'z\,[{\rm kpc}]')
    
    ax._raw.set_aspect('equal')
    ax._raw.grid(False)
    ax._raw.set_facecolor('k')
    ax._raw.tick_params(axis='both', color='w', width=1, length=5)
    axs.label_outer()
    
# plot.savefig(out_fig_dir/'model-edge-on.pdf')

In [ ]:
fig, axs = plot.subplots((1,3), share=False, space=0.02, subsize=(5., 5.), 
                         margin=[0.1, 0.1, 0.1, 0.1], layout='none')
axs_f = axs.flat

norm=mcolors.Normalize(vmin=0,vmax=60)

xs, ys = recon.x_cs, recon.y_cs
v_x, v_y = fields_3D['edge_on/IGM']['v_x/median', 'v_y/median']
v_x, v_y = v_x.T, v_y.T
v_x1, v_y1, rs, incs, R_h, e_thetas = recon.data['v_xs', 'v_ys', 'rs', 'incs', 'R_h', 'e_thetas']
dv_x, dv_y = v_x1 - v_x, v_y1 - v_y

v_theta = v_x * e_thetas[0] + v_y * e_thetas[1]
dv_x1, dv_y1 = v_theta * e_thetas[0], v_theta * e_thetas[1]
v_x2, v_y2 = v_x1 + dv_x1, v_y1 + dv_y1
dv_x, dv_y = v_x2 - v_x, v_y2 - v_y

stream_kw = dict(cmap='rainbow',norm=norm,linewidth=1.,
                 density=1,
                 #density=2,
                 arrowsize=.8)
ax = axs_f[0]
v_norm = np.hypot(v_x2, v_y2)
strm= ax._raw.streamplot(xs, ys, v_x2, v_y2, color=v_norm, **stream_kw)

ax = axs_f[1]
v_norm = np.hypot(v_x, v_y)
strm= ax._raw.streamplot(xs, ys, v_x, v_y, color=v_norm, **stream_kw)

ax = axs_f[2]
v_norm = np.hypot(dv_x, dv_y)
dv = np.hypot(dv_x, dv_y)
ax._raw.pcolormesh(xs, ys, dv, cmap='rainbow', norm=norm)

for ax in axs_f:
    ax.lim([-675, 675], [-675, 675])#label(r'x\,[{\rm kpc}]', r'y\,[{\rm kpc}]')\    
    ax._raw.set_aspect('equal')
    ax._raw.grid(False)
    ax._raw.set_facecolor('k')
    ax._raw.tick_params(axis='both', color='w', width=1, length=5)
    axs.label_outer()

In [ ]:
samp_key = 'z0/lm10'

prop_maps = field2D_data[samp_key]
header = field2D_data['header']
ecosys_def = ecosys_defs[samp_key]
R_h, V_h, H = ecosys_def['R_h', 'V_h', 'H']

v_theta =  prop_maps['v_theta'] 
ys = v_theta / V_h

theta_edges, r_edges = header['theta_edges', 'r_edges']
inc_edges = -theta_edges + np.pi/2
s_inc_edges = np.sin(inc_edges)
s_inc_cs = .5 * (s_inc_edges[:-1] + s_inc_edges[1:])
inc_cs = np.arcsin(s_inc_cs)
r_cs = .5 * (r_edges[:-1] + r_edges[1:])
xs = r_cs / R_h

sel = xs < 4.
xs, ys = xs[sel], ys[:, sel]
# y_minor = all_data[samp_key]['v_r/minor'] / V_h
# r_es_1d = all_data['header']['r_edges']
# r_cs_1d = .5 * (r_es_1d[1:] + r_es_1d[:-1])
# xs_1d = r_cs_1d / R_h

In [ ]:
vtheta_fit = VthetaFit(inc_cs, xs, ys)
vtheta_fit.find_optim()
vtheta_fit.opt_map

In [ ]:
fig, axs = plot.subplots((2,2), share=False, space=0.25, subsize=(5.0, 5.0), margin=[0.1, 0.1, 0.1, 0.1], layout='none')
axs_f = axs.flat

y_map = ys 
y_ini = vtheta_fit.init_map.y_at(vtheta_fit.theta_map, vtheta_fit.x_map)
y_opt = vtheta_fit.opt_map.y_at(vtheta_fit.theta_map, vtheta_fit.x_map)

kw = dict(cmap='RdBu_r', vmin=-0.25, vmax=0.25)

ax = axs_f[0]
ax._raw.pcolormesh(inc_cs, xs, y_map.T, **kw)

ax = axs_f[1]
ax._raw.pcolormesh(inc_cs, xs, y_ini.T, **kw)

ax = axs_f[2]
ax._raw.pcolormesh(inc_cs, xs, y_opt.T, **kw)

ax = axs_f[3]
ax._raw.pcolormesh(inc_cs, xs, (y_opt-y_map).T, **kw)

axs.lim(y=[0., 5.5])

In [ ]:
fig, axs = plot.subplots((2,2), share=False, space=0.25, subsize=(7.5, 5.0), margin=[0.1, 0.1, 0.1, 0.1], layout='none')
axs_f = axs.flat

xs_show = [2., 0.05]#.01, .075, .15, .2, .25, .3, .35, .5, .75, 1., 1.2, 1.3, 1.5, 2., 2.5, 3., 4., 5.5, 7.5, 9.5]
cs = mpl.cm.Reds(np.linspace(.1,.9,len(xs_show)))
for i_x, x_show in enumerate(xs_show):
    axs_f.c(cs[i_x])
    
    ix = np.abs(xs - x_show).argmin()
    
    ax = axs_f[0]
    ys_show = ys[:, ix]    
    ax.plot(s_inc_cs, ys_show, lw=2)
    
    ax = axs_f[1]
    theta = np.arcsin(s_inc_cs)
    ax.plot(theta, ys_show, lw=2)
    
    if i_x == 0:
        p1, p2 = 1.25, 1.5
        yps = -0.075 * np.abs(theta)**p1 * (np.pi/2 - np.abs(theta))**p2/np.abs(np.pi/4.)**p1 / (np.pi/2 - np.abs(np.pi/4.))**p2
        ax.plot(theta, yps, c='k', ls='--')
    if i_x == 1:
        p1, p2 = 2.75, 1.2
        yps = 0.2 * np.abs(theta)**p1 * (np.pi/2 - np.abs(theta))**p2/np.abs(1.2)**p1 / (np.pi/2 - np.abs(1.2))**p2
        ax.plot(theta, yps, c='k', ls='--')
    
thetas_show = [np.pi/4, 1.2]
for i_t, theta_show in enumerate(thetas_show):
    axs_f.c('b')
    
    it = np.abs(inc_cs - theta_show).argmin()
    
    ax = axs_f[2]
    ys_show = ys[it, :]    
    ax.plot(xs, ys_show, lw=2)
    
    if i_t == 0:
        x1 = 2.
        yps = -0.075*np.exp(-(np.abs(xs-x1)/15.)**1.5)
        yps[xs<x1] = -0.075*np.exp(-(np.abs(xs[xs<x1]-x1)/1.)**1.5)
        ax.plot(xs, yps, c='k', ls='--')
    elif i_t == 1:
        x1 = .05
        yps = .2*np.exp(-(np.abs(xs-x1)/.305)**1.01)
        yps[xs<=x1] = .2*np.exp(-(np.abs(xs[xs<x1]-x1)/0.02)**2.)
        ax.plot(xs, yps, c='k', ls='--')
    
axs_f[2].lim([0.001, 1.5]).lim(y=[-.2, .2])#.scale('log')

In [ ]:
recon = EdgeOnRecon(field3D_data['header']['bin_edges/IGM']['x', 'y'], 
                    vout_fit.opt_map, ecosys_def)
recon.run()

In [ ]:
incs, rs, R_h, V_h, e_thetas = recon.data['incs', 'rs', 'R_h', 'V_h', 'e_thetas']
v_x, v_y = field3D_data['edge_on/IGM']['v_x', 'v_y']
v_theta = v_x * e_thetas[0] + v_y * e_thetas[1]

In [ ]:
lo, hi = np.deg2rad([-30., -28.])

In [ ]:
sel = (incs >= lo) & (incs < hi)
xs, ys = rs[sel]/R_h,  v_theta[sel] /V_h

In [ ]:
y_sum, x_e = np.histogram(xs, weights=ys, bins=50, range=(0., 4.5))
n_y, _ = np.histogram(xs, bins=50)
y_avg = y_sum / n_y.clip(.1)

In [ ]:
fig, ax = plot.subplots(1, figsize=4.5, margin=[0.1, 0.1, 0.1, 0.1], layout='none')

x_c = .5 * (x_e[1:] + x_e[:-1])
ax.plot(x_c, y_avg, c='k', lw=2)

In [ ]:

xs, ys = recon.x_cs, recon.y_cs
v_x, v_y = field3D_data['edge_on/IGM']['v_x', 'v_y']
v_x1, v_y1, rs, incs, R_h, e_thetas = recon.data['v_xs', 'v_ys', 'rs', 'incs', 'R_h', 'e_thetas']

x_rs = rs/R_h
dv_x, dv_y = v_x1 - v_x, v_y1 - v_y
v_theta = v_x * e_thetas[0] + v_y * e_thetas[1]

fig, ax = plot.subplots(1, figsize=6.5, margin=[0.1, 0.1, 0.1, 0.1], layout='none')

lt = ax._raw.pcolormesh(xs, ys, v_theta/V_h, cmap='bwr', vmin=-.25, vmax=.25)

ax._raw.set_aspect('equal')
fig.colorbar(lt, ax=ax._raw, pad=.1)
ax.lim([-675, 675], [-675, 675])

In [ ]:
samp_key = 'z0/lm10'

prop_maps = field2D_data[samp_key]
header = field2D_data['header']
ecosys_def = ecosys_defs[samp_key]
R_h, V_h, H = ecosys_def['R_h', 'V_h', 'H']

v_r =  prop_maps['v_r'] 
ys = v_r / V_h

theta_edges, r_edges = header['theta_edges', 'r_edges']
inc_edges = -theta_edges + np.pi/2
s_inc_edges = np.sin(inc_edges)
s_inc_cs = .5 * (s_inc_edges[:-1] + s_inc_edges[1:])
r_cs = .5 * (r_edges[:-1] + r_edges[1:])
xs = r_cs / R_h

y_minor = all_data[samp_key]['v_r/minor'] / V_h
r_es_1d = all_data['header']['r_edges']
r_cs_1d = .5 * (r_es_1d[1:] + r_es_1d[:-1])
xs_1d = r_cs_1d / R_h

In [ ]:
theta_edges

In [ ]:
inc_edges

In [ ]:
fig, axs = plot.subplots((2,2), share=False, space=0.25, subsize=(7.0, 5.0), margin=[0.1, 0.1, 0.1, 0.1], layout='none')
axs_f = axs.flat

xs_show = [.005, .1, .25, .3, 1.2, 1.3, 1.5]
cs = mpl.cm.viridis(np.linspace(0,1,len(xs_show)))
for i_x, x_show in enumerate(xs_show):
    axs_f.c(cs[i_x])
    
    ix = np.abs(xs - x_show).argmin()
    
    ax = axs_f[0]
    ys_show = ys[:, ix]    
    ax.plot(s_inc_cs, ys_show, lw=2)
    
    ax = axs_f[1]
    theta = np.arcsin(s_inc_cs)
    if i_x == 2:
        ax.plot(theta, ys_show, lw=3, c='k', zorder=10)
        xps = np.linspace(-np.pi/2, np.pi/2, 128)
        yps = .55 * np.exp( -(np.pi/2-np.abs(xps))**2.5/.25 ) - \
            .125 * np.exp( -(np.abs(xps))**2.5/.5 )
        ax.plot(xps, yps, lw=2, ls='--', c='r')
    else:
    
        ax.plot(theta, ys_show, lw=2)
    
    
ax = axs_f[1]
ax.lim([-np.pi/2, np.pi/2], [-0.6, 0.96])
ax.label(r'\sin i', r'v_{\rm r}/V_{\rm h}')

ax = axs_f[2]
v_r_cs = (v_r[0] + v_r[-1])*.5 #- H * r_cs
y_cs = v_r_cs / V_h

y_H = H * r_cs * 1. / V_h
y_ins = -vin_fit.opt_pf.y_at(xs) + y_H
dys = y_cs - y_H

#i_loc = len(s_inc_cs)//2-1
#v_r_ds = v_r[i_loc:i_loc+2].mean(0)
#y_ds = v_r_ds / V_h
#y_cs = y_ds
#y_ds = H * r_cs / V_h

ax.plot(xs, y_cs, lw=2)
ax.plot(xs_1d, y_minor, lw=2, ls='--', c='g')
ax.plot(xs, y_H, lw=2, c='r')
ax.plot(xs, dys, lw=2, c='k')
ax.lim([0.01, 3.5], [-.15, .75])

ax = axs_f[3]
ax.plot(xs, y_cs, lw=2)
ax.plot(xs_1d, y_minor, lw=2, ls='--', c='g')
ax.plot(xs, dys, lw=2, c='k')
ax.lim([0.01, 25.], [0.01, 1.5])
ax.scale('log')


# 1D profiles

## Regimes

In [ ]:
f_in = sim_dir / '1Dprofiles_interp.hdf5'
fields_1D = h5.File.load_from(f_in)

header = fields_1D['header']
theta_edges, r_edges = header['theta_edges', 'r_edges']
r_cs = .5 * (r_edges[:-1] + r_edges[1:])

In [ ]:
keys_list = [
    ('z0', 'lm9'),
    ('z0', 'lm10'),
    ('z0', 'lm11'),
    ('z0', 'lm11_hiBH'),
    ('z1', 'lm10'),
    ('z2', 'lm10'),
]
for i_k, (z_key, lm_key) in enumerate(keys_list):
    profs = fields_1D[z_key][lm_key]
    v_phi = profs['v_phi/disk/median']
    arg = v_phi.argmax()
    v_dst = v_phi[arg] * .5
    i_r = ((np.arange(len(v_phi)) > arg) & (v_phi <= v_dst)).nonzero()[0][0]
    i_l = i_r - 1
    assert v_phi[i_l] >= v_dst
    assert v_phi[i_r] <= v_dst
    R_ISM = np.interp(v_dst, v_phi[i_l:i_r+1][::-1], r_cs[i_l:i_r+1][::-1])
    
    v_r = profs['v_r/disk/median']
    i_l = (v_r < 0.).nonzero()[0][-1]
    i_r = i_l + 1
    assert v_r[i_l] <= 0.
    assert v_r[i_r] >= 0.
    R_CGM = np.interp(0., v_r[i_l:i_r+1], r_cs[i_l:i_r+1])
    print(f'{z_key=}, {lm_key=}, {v_dst=}, {R_ISM=}, {R_CGM=}')

## Plots

In [ ]:
f_in = sim_dir / '1Dprofiles_interp.hdf5'
fields_1D = h5.File.load_from(f_in)

header = fields_1D['header']
theta_edges, r_edges = header['theta_edges', 'r_edges']

In [ ]:
inc_edges = -theta_edges + np.pi/2
absinc_edges = inc_edges[25:]
absinc_cs = .5 * (absinc_edges[:-1] + absinc_edges[1:])

absinc_cs[-1] = np.pi/2.
absinc_cs[0] = 0.

r_cs = .5 * (r_edges[:-1] + r_edges[1:])

In [ ]:
y_keys = 'lg_n', 'v_phi', 'lambda', 'lg_T', 'v_r', 'I'
y_labs=[r'\log n\, [{\rm cm}^{-3}]',
           r'v_\phi\, [{\rm km/s}]',
           r'\log\,\lambda',
           r'\log T\, [{\rm K}]',
           r'v_r\, [{\rm km/s}]',
           r'I\, [{\rm M_\odot\,yr^{-1}sr^{-1}}]']

In [ ]:
ecosys = ecosys_infos['z0/lm10']
profs = fields_1D['z0/lm10']

In [ ]:
fig, axs = plot.subplots((2,3), share=False, 
                         space=(.3, .035), 
                         subsize=(5*.9, 2.5*.9), 
                         margin=[0.02, 0.05, 0.12, 0.05], 
                         layout='none')
axs_f = axs.flat

y_lims = [-7.9, 0.95], [-25., 165.], [-2.75, -.5], [3.85, 6.45], [-50., 185.], [-.3, .8]
x_nodes = [1., ecosys.R_ISM, ecosys.R_CGM, 2200]
regime_alphas = .75, .5, .25
regime_cs = cs_named['blue', 'green','orange']

# reduc_key = 'median'
reduc_key = 'mean'

for i in range(2):
    for j in range(3):
        data_id = 3*i + j
        ax = axs[i,j]
        minor, major = profs[y_keys[data_id]][f'polar/{reduc_key}', f'disk/{reduc_key}']
        if data_id == 2:
            major, minor = np.log10(major.clip(1.0e-10)), np.log10(minor.clip(1.0e-10))
        ax.plot(r_cs, major, lw=3.5, label=r'$\text{Disk plane}$')
        ax.plot(r_cs, minor, lw=1, label=r'$\text{Outflow cone}$')
        
        for k in range(3):
            y_lb, y_ub = y_lims[3*i+j] 
            y_ub = y_lb + (y_ub - y_lb) * .2
            ax.fill_between([x_nodes[k], x_nodes[k+1]], [y_lb]*2, [y_ub]*2, 
                            color=regime_cs[k], alpha=regime_alphas[k], 
                            lw=0)
        
kw = dict(lw=1.5, c='grey', ls=(0,(1.5,1.5)))
x = np.logspace(1., 1.5, 10)
y = -2*np.log10(x)
y = y - y[0] - 4.
axs_f[0].plot(x, y, **kw)\
    .leg(loc='ur', handlelength=1.2)

y = x * 1.
y = np.log10(y/y[0] * 1.0e-2)

axs_f[2].plot(x, y, **kw)
axs[1].label(r'r\, [{\rm kpc}]')

axs.lim([1., 2000]).scale('log')    
for i, ax in enumerate(axs_f):
    ax.label(y=y_labs[i]).lim(y=y_lims[i])
for ax in axs[0]:
    ax._raw.set_xticklabels([])

plot.savefig(out_fig_dir/'mass10z0_r.pdf')

In [ ]:
fig, axs = plot.subplots((2,3), share=False, 
                         space=(.3, .035), 
                         subsize=(5*.9, 2.5*.9), 
                         margin=[0.02, 0.05, 0.12, 0.05], 
                         layout='none')
axs_f = axs.flat

y_lims = [-7.9, 0.95], [-25., 165.], [-2.75, -.75], [3.85, 6.25], [-50., 185.], [-.3, .6]
regime_labs = r'$r=10\,{\rm kpc}$', r'$r=100\,{\rm kpc}$', r'$r=1000\,{\rm kpc}$'
regime_cs = cs_named['blue', 'green','orange']
for i in range(2):
    for j in range(3):    
        data_id = 3*i + j
        ax = axs[i,j]
        for i_y, rkey in enumerate(['r10', 'r100', 'r1000']):
            x = np.rad2deg(absinc_cs)
            y = profs[y_keys[data_id]][rkey]['median']
            if data_id == 2:
                y = np.log10(y.clip(1.0e-10))            
            ax.plot(x,y, lw=3.5, label=regime_labs[i_y], c=regime_cs[i_y])

#kw = dict(lw=1.5, c='grey', ls=(0,(1.5,1.5)))
#axs_f[2].plot(x, y, **kw)
axs_f[0].leg(loc='ur', handlelength=1.2, labelspacing=0, labelcolor='linecolor')
axs[1].label(r'\left|\theta\right|\,[{\rm deg}]')

axs.lim([0., 90.])
for i, ax in enumerate(axs_f):
    ax.label(y=y_labs[i]).lim(y=y_lims[i])
    tks = np.arange(0,91,30)
    ax._raw.set_xticks(tks)
for ax in axs[0]:
    ax._raw.set_xticklabels([])

plot.savefig(out_fig_dir/'mass10z0_theta.pdf')

In [ ]:
ecosys_list = ecosys_infos['z0']['lm9', 'lm10', 'lm11']
profs = fields_1D['z0']['lm9', 'lm10', 'lm11']

In [ ]:
fig, axs = plot.subplots((2,3), share=False, 
                         space=(.3, .035), 
                         subsize=(5*.9, 2.5*.9), 
                         margin=[0.02, 0.05, 0.12, 0.05], 
                         layout='none')
axs_f = axs.flat

m_labs = (r'$M_* = 10^{9}\,{\rm M}_\odot$', 
          r'$M_* = 10^{10}\,{\rm M}_\odot$', 
          r'$M_* = 10^{11}\,{\rm M}_\odot$')
m_cs = cs_named['b', 'g', 'orange']
y_lims = [-7.9, 2.05], [-25., 255.], [-2.75, -.425], [3.85, 7.35], [-85., 395.], [-1.2, 4.25]
reduc_key = 'mean'
# reduc_key = 'median'

for i in range(2):
    for j in range(3):
        data_id = 3*i + j
        ax = axs[i,j]
        
        for i_m, ecosys in enumerate(ecosys_list):
            r_ism, r_cgm = ecosys.R_ISM, ecosys.R_CGM
            m_c = m_cs[i_m]
            y_lb, y_ub = y_lims[data_id]
            dy = (y_ub - y_lb)
            arrow_prop = {'arrowstyle': '-|>','lw': 1., 'fc': m_c, 'ec': m_c}
            ax._raw.annotate('', (r_ism, y_ub-dy*.2), xytext=(r_ism, y_ub-dy*.05), 
                             arrowprops=arrow_prop|{'lw': 1.})
            ax._raw.annotate('', (r_cgm, y_ub-dy*.22), xytext=(r_cgm, y_ub-dy*.05), 
                             arrowprops=arrow_prop|{'lw': 3.5})
            
            minor, major = profs[i_m][y_keys[data_id]][f'polar/{reduc_key}', f'disk/{reduc_key}']
            if data_id == 2:
                major, minor = np.log10(major.clip(1.0e-10)), np.log10(minor.clip(1.0e-10))
            if data_id == 5:
                if i_m == 0:
                    major, minor = major * 10., minor * 10.
                elif i_m == 1:
                    major, minor = major * 2, minor * 2
            ax.plot(r_cs,major, lw=3.5, label=m_labs[i_m], c=m_c)
            ax.plot(r_cs,minor, lw=1.25, c=m_c)
            
kw = dict(lw=1.5, c='grey', ls=(0,(1.5,1.5)))
x = np.logspace(2., 2.5, 10)
y = -2*np.log10(x)
y = y - y[0] - 2.
axs_f[0].plot(x, y, **kw)\
    .leg(loc='ll', handlelength=1.2, labelspacing=-.2, 
         labelcolor='linecolor', fontsize=13.)

x = np.logspace(1., 1.5, 10)
y = x * 1.
y = np.log10(y/y[0] * 1.0e-2)

axs_f[2].plot(x, y, **kw)
axs[1].label(r'r\, [{\rm kpc}]')

for i, ax in enumerate(axs_f):
    ax.label(y=y_labs[i]).lim([1., 2000], y_lims[i]).scale('log')
for ax in axs[0]:
    ax._raw.set_xticklabels([])    

# plot.savefig(out_fig_dir/'mdependence2_r.pdf')
plot.savefig(out_fig_dir/f'mdependence2_r_{reduc_key}.pdf')

In [ ]:
fig, axs = plot.subplots((2,3), share=False, 
                         space=(.3, .035), 
                         subsize=(5*.9, 2.5*.9), 
                         margin=[0.02, 0.05, 0.12, 0.05], 
                         layout='none')
axs_f = axs.flat

m_cs = cs_named['b', 'g', 'orange']
y_lims = [-7.9, 0.25], [-30., 225.], [-2.75, -.75], [3.85, 7.1], [-50., 325.], [-1., 3.]
r_kws = {'lw': 3.5}, {'lw': 1.25}, {'lw': 1.25, 'ls': (0, (2.5,1.5))}
for i in range(2):
    for j in range(3):    
        data_id = 3*i + j
        ax = axs[i,j]
        for i_m in range(3):
            m_c = m_cs[i_m]
            for i_r, rkey in enumerate(['r10', 'r100', 'r1000']):
                x = np.rad2deg(absinc_cs)
                y = profs[i_m][y_keys[data_id]][rkey]['median']
                if data_id == 2:
                    y = np.log10(y.clip(1.0e-10)) 
                if data_id == 5:
                    if i_m == 0:
                        y = y * 10.
                    elif i_m == 1:
                        y = y * 2.
                ax.plot(x, y, c=m_c, **r_kws[i_r])
        
axs_f[0].leg(loc='ur', handlelength=1.2, labelspacing=0, labelcolor='linecolor')
#axs_f[2].plot(x, y, **kw)
axs[1].label(r'\left|\theta\right|\,[{\rm deg}]')

axs.lim([0., 90.])
for i, ax in enumerate(axs_f):
    ax.label(y=y_labs[i]).lim(y=y_lims[i])
    tks = np.arange(0,91,30)
    ax._raw.set_xticks(tks)
for ax in axs[0]:
    ax._raw.set_xticklabels([])

plot.savefig(out_fig_dir/'mdependence2_theta.pdf')

In [ ]:
ecosys_list = ecosys_infos['z2/lm10', 'z1/lm10', 'z0/lm10']
profs = fields_1D['z2/lm10', 'z1/lm10', 'z0/lm10']

In [ ]:
fig, axs = plot.subplots((2,3), share=False, 
                         space=(.3, .035), 
                         subsize=(5, 2.5*.9), 
                         margin=[0.02, 0.05, 0.12, 0.125], 
                         layout='none')
axs_f = axs.flat

z_labs = (r'$z=2$', 
          r'$z=1$', 
          r'$z=0$')
z_cs = cs_named['orange', 'g', 'b']
y_lims = [-7.9, 2.55], [-25., 255.], [-2.75, -.25], [3.55, 7.15], [-125., 355.], [-1.65, 4.55]

for i in range(2):
    for j in range(3):
        data_id = 3*i + j
        ax = axs[i,j]
        
        for i_z, ecosys in enumerate(ecosys_list):
            r_ism, r_cgm = ecosys.R_ISM, ecosys.R_CGM
            z_c = z_cs[i_z]
            y_lb, y_ub = y_lims[data_id]
            dy = (y_ub - y_lb)
            arrow_prop = {'arrowstyle': '-|>','lw': 1., 'fc': z_c, 'ec': z_c}
            ax._raw.annotate('', (r_ism, y_ub-dy*.2), xytext=(r_ism, y_ub-dy*.05), 
                             arrowprops=arrow_prop|{'lw': 1.})
            ax._raw.annotate('', (r_cgm, y_ub-dy*.22), xytext=(r_cgm, y_ub-dy*.05), 
                             arrowprops=arrow_prop|{'lw': 3.5})
        
            minor, major = profs[i_z][y_keys[data_id]]['polar/median', 'disk/median']
            if data_id == 2:
                major, minor = np.log10(major.clip(1.0e-10)), np.log10(minor.clip(1.0e-10))    
        
            ax.plot(r_cs,major, lw=3.5, label=z_labs[i_z], c=z_c)
            ax.plot(r_cs,minor, lw=1.25, c=z_c)
            
kw = dict(lw=1.5, c='grey', ls=(0,(1.5,1.5)))
x = np.logspace(2., 2.5, 10)
y = -2*np.log10(x)
y = y - y[0] - 2.
axs_f[0].plot(x, y, **kw)\
    .leg(loc='ll', handlelength=1.2, labelspacing=0., 
         labelcolor='linecolor', fontsize=13.)

x = np.logspace(1., 1.5, 10)
y = x * 1.
y = np.log10(y/y[0] * 1.0e-2)

axs_f[2].plot(x, y, **kw)
axs[1].label(r'r\, [{\rm kpc}]')

for i, ax in enumerate(axs_f):
    ax.label(y=y_labs[i]).lim([1., 2000], y_lims[i]).scale('log')
for ax in axs[0]:
    ax._raw.set_xticklabels([])    

plot.savefig(out_fig_dir/'zdependence2_r.pdf')

In [ ]:
fig, axs = plot.subplots((2,3), share=False, 
                         space=(.3, .035), 
                         subsize=(5*.9, 2.5*.9), 
                         margin=[0.02, 0.05, 0.12, 0.05], 
                         layout='none')
axs_f = axs.flat

z_cs = cs_named['orange', 'g', 'b']
y_lims = [-7.9, 0.25], [-35., 165.], [-3.55, -.55], [3.25, 6.75], [-75., 275.], [-0.75, 2.95]

r_kws = {'lw': 3.5}, {'lw': 1.25}, {'lw': 1.25, 'ls': (0, (2.5,1.5))}
for i in range(2):
    for j in range(3):    
        data_id = 3*i + j
        ax = axs[i,j]
        for i_z in range(3):
            z_c = z_cs[i_z]
            for i_r, rkey in enumerate(['r10', 'r100', 'r1000']):
                x = np.rad2deg(absinc_cs)
                y = profs[i_z][y_keys[data_id]][rkey]['median']
                if data_id == 2:
                    y = np.log10(y.clip(1.0e-10)) 
                ax.plot(x, y, c=z_c, **r_kws[i_r])
        
axs_f[0].leg(loc='ur', handlelength=1.2, labelspacing=0, labelcolor='linecolor')
axs_f[2].plot(x, y, **kw)
axs[1].label(r'\left|\theta\right|\,[{\rm deg}]')

axs.lim([0., 90.])
for i, ax in enumerate(axs_f):
    ax.label(y=y_labs[i]).lim(y=y_lims[i])
    tks = np.arange(0,91,30)
    ax._raw.set_xticks(tks)
for ax in axs[0]:
    ax._raw.set_xticklabels([])

plot.savefig(out_fig_dir/'zdependence2_theta.pdf')

In [ ]:
ecosys_list = ecosys_infos['z0']['lm11_hiBH', 'lm11']
profs = fields_1D['z0']['lm11_hiBH', 'lm11']

In [ ]:
fig, axs = plot.subplots((2,3), share=False, 
                         space=(.3, .035), 
                         subsize=(5*.9, 2.5*.9), 
                         margin=[0.02, 0.05, 0.12, 0.05], 
                         layout='none')
axs_f = axs.flat
y_lims = [-7.9, 2.05], [-25., 285.], [-2.75, -.425], [3.85, 7.25], [-85., 455.], [-1.2, 5.55]

m_cs = cs_named['r', 'b']
m_labs = r'${\rm High}\,M_{\rm BH}$', r'${\rm Low}\,M_{\rm BH}$'
for i in range(2):
    for j in range(3):
        data_id = 3*i + j
        ax = axs[i,j]
        
        for i_m, ecosys in enumerate(ecosys_list):
            r_ism, r_cgm = ecosys.R_ISM, ecosys.R_CGM
            m_c = m_cs[i_m]
            y_lb, y_ub = y_lims[data_id]
            dy = (y_ub - y_lb)
            arrow_prop = {'arrowstyle': '-|>','lw': 1., 'fc': m_c, 'ec': m_c}
            ax._raw.annotate('', (r_ism, y_ub-dy*.2), xytext=(r_ism, y_ub-dy*.05), 
                             arrowprops=arrow_prop|{'lw': 1.})
            ax._raw.annotate('', (r_cgm, y_ub-dy*.22), xytext=(r_cgm, y_ub-dy*.05), 
                             arrowprops=arrow_prop|{'lw': 3.5})
            
            minor, major = profs[i_m][y_keys[data_id]]['polar/median', 'disk/median']
            if data_id == 2:
                major, minor = np.log10(major.clip(1.0e-10)), np.log10(minor.clip(1.0e-10))
            ax.plot(r_cs,major, lw=3.5, c=m_c, label=m_labs[i_m])
            ax.plot(r_cs,minor, c=m_c, lw=1)
        
kw = dict(lw=1.5, c='grey', ls=(0,(1.5,1.5)))
x = np.logspace(2., 2.5, 10)
y = -2*np.log10(x)
y = y - y[0] - 2.
axs_f[0].plot(x, y, **kw)\
    .leg(loc='ll', handlelength=1.2, labelspacing=0., 
         labelcolor='linecolor', fontsize=13.5)

x = np.logspace(1., 1.5, 10)
y = x * 1.
y = np.log10(y/y[0] * 1.0e-2)
axs_f[2].plot(x, y, **kw)
axs[1].label(r'r\, [{\rm kpc}]')

axs.lim([1., 2000]).scale('log')    
for i, ax in enumerate(axs_f):
    ax.label(y=y_labs[i]).lim(y=y_lims[i])
for ax in axs[0]:
    ax._raw.set_xticklabels([])

plot.savefig(out_fig_dir/'highmassSMBH_r.pdf')

In [ ]:
fig, axs = plot.subplots((2,3), share=False, 
                         space=(.3, .035), 
                         subsize=(5*.9, 2.5*.9), 
                         margin=[0.02, 0.05, 0.12, 0.05], 
                         layout='none')
axs_f = axs.flat

m_cs = cs_named['r', 'b']
y_lims = [-7.9, 0.25], [-30., 280.], [-2.75, -.75], [3.85, 7.25], [-85., 345.], [-1., 4.75]
r_kws = {'lw': 3.5}, {'lw': 1.25}, {'lw': 1.25, 'ls': (0, (2.5,1.5))}
for i in range(2):
    for j in range(3):    
        data_id = 3*i + j
        ax = axs[i,j]
        
        for i_m, ecosys in enumerate(ecosys_list):
            m_c = m_cs[i_m]
            for i_r, rkey in enumerate(['r10', 'r100', 'r1000']):
                x = np.rad2deg(absinc_cs)
                y = profs[i_m][y_keys[data_id]][rkey]['median']
                if data_id == 2:
                    y = np.log10(y.clip(1.0e-10)) 
                ax.plot(x, y, **r_kws[i_r], c=m_c)
        
axs_f[0].leg(loc='ur', handlelength=1.2, labelspacing=0, labelcolor='linecolor')
axs_f[2].plot(x, y, **kw)
axs[1].label(r'\left|\theta\right|\,[{\rm deg}]')

axs.lim([0., 90.])
for i, ax in enumerate(axs_f):
    ax.label(y=y_labs[i]).lim(y=y_lims[i])
    tks = np.arange(0,91,30)
    ax._raw.set_xticks(tks)
for ax in axs[0]:
    ax._raw.set_xticklabels([])

plot.savefig(out_fig_dir/'highmassSMBH_theta.pdf')

## Field computation

In [ ]:
f_in = sim_dir / '2Dmaps_individual_interp.hdf5'
fields_2D = h5.File.load_from(f_in)

In [ ]:
header = fields_2D['header'].copy()

In [ ]:
r_es, theta_es = header['r_edges', 'theta_edges']
r_cs, theta_cs = .5 * (r_es[:-1] + r_es[1:]), .5 * (theta_es[:-1] + theta_es[1:])

In [ ]:
r_args = [np.argmin(np.abs(r_cs-r_dst)) for r_dst in (10, 100, 1000)]
r_args

In [ ]:
keys_list = [
    ('z0', 'lm9'),
    ('z0', 'lm10'),
    ('z0', 'lm11'),
    ('z0', 'lm11_hiBH'),
    ('z1', 'lm10'),
    ('z2', 'lm10'),
]
out = {'header': header}
for z_key, m_key in keys_list:
    dsets = fields_2D[z_key][m_key]
    print(z_key, m_key)
    for k, v in dsets.items():
        print(' - ', k, v.shape)
        v_polar = v[:, :, [0,-1]]
        v_disk = v[:, :, [24,25]]
        v_10 = v[:, 9, :]
        v_100 = v[:, 51, :]
        v_1000 = v[:, 87, :]
        sub_vs = [v_polar, v_disk, v_10, v_100, v_1000]
        sub_ks = 'polar', 'disk', 'r10', 'r100', 'r1000'
        for sub_v, sub_k in zip(sub_vs, sub_ks):
            out.setdefault(z_key, {}).setdefault(m_key, {}).setdefault(k, {})[sub_k] = sub_v

In [ ]:
f_out = sim_dir/ '1Dprofiles_individual_interp.hdf5'
h5.File.dump_to(f_out, out, f_flag='w')

In [ ]:
keys_list = [
    ('z0', 'lm9'),
    ('z0', 'lm10'),
    ('z0', 'lm11'),
    ('z0', 'lm11_hiBH'),
    ('z1', 'lm10'),
    ('z2', 'lm10'),
]
out = {'header': header}
for z_key, m_key in keys_list:
    dsets = fields_2D[z_key][m_key]
    print(z_key, m_key)
    for k, v in dsets.items():
        print(' - ', k, v.shape)
        v_polar = v[:, :, [0,-1]]
        v_disk = v[:, :, [24,25]]
        v_10 = v[:, 9, :]
        v_100 = v[:, 51, :]
        v_1000 = v[:, 87, :]
        sub_vs = [v_polar, v_disk, v_10, v_100, v_1000]
        axes = [(0,2), (0,2), (0,), (0,), (0,)]
        sub_ks = 'polar', 'disk', 'r10', 'r100', 'r1000'
        folds = False, False, True, True, True
        for sub_v, axis, sub_k, fold in zip(sub_vs, axes, sub_ks, folds):
            if fold:
                n_col = sub_v.shape[1]//2
                sub_v = np.concatenate([sub_v[:, :n_col][:, ::-1], sub_v[:, n_col:]], axis=0)
            median = np.median(sub_v, axis=axis)
            q16 = np.quantile(sub_v, 0.16, axis=axis)
            q84 = np.quantile(sub_v, 0.84, axis=axis)
            mean = np.mean(sub_v, axis=axis)
            stddev = np.std(sub_v, axis=axis)
            out.setdefault(z_key, {}).setdefault(m_key, {}).setdefault(k, {})[sub_k] = {
                'median': median,
                'mean': mean,
                'stddev': stddev,
                'q16': q16,
                'q84': q84,
            }

In [ ]:
f_out = sim_dir/ '1Dprofiles_interp.hdf5'
h5.File.dump_to(f_out, out, f_flag='w')

In [ ]:
h5.File.ls_from(f_out)

## Additional profiles

In [ ]:
# v_r, spherically averaged

In [ ]:
sph_avg_pfs = h5.File.load_from(sim_dir/'1D_spherical_profiles.hdf5')

In [ ]:
v_r = sph_avg_pfs['z0/lm10']['DM_v_r']
r_es = sph_avg_pfs['header']['r_edges']
r_cs = .5 * (r_es[:-1] + r_es[1:])
v_r_med = np.mean(v_r, axis=0)

In [ ]:
fig, ax = plot.subplots(1, figsize=4.5, margin=[0.1, 0.1, 0.1, 0.1], layout='none')

ax.plot(r_cs, v_r_med, lw=2)

ax.lim([1., 1500.], [-5., 100.]).scale('log')

# 2D maps

## Plots

In [ ]:
f_in = sim_dir / '2Dmaps_interp.hdf5'
fields_2D = h5.File.load_from(f_in)

In [ ]:
header = fields_2D['header']
theta_edges, r_edges = header['theta_edges', 'r_edges']
#theta_edges = np.pi/2. - theta_edges
theta_ms, r_ms = np.meshgrid(theta_edges,r_edges,indexing='ij')

In [ ]:
y_confs = {
    'lg_n': {
        'y_label': r'$\log n\, [{\rm cm}^{-3}]$',
        'cmap': 'gnuplot2',
    },
    'lg_T': {
        'y_label': r'$\log T \,[{\rm K}]$',
        'cmap': 'coolwarm',
    },
    'v_phi': {
        'y_label': r'$v_\phi\, [{\rm km/s}]$',
        'cmap': 'RdBu_r',
    },
    'v_r': {
        'y_label': r'$v_r\, [{\rm km/s}]$',
        'cmap': 'RdBu_r',
    },
    'lambda': {
        'y_label': r'$\lambda$',
        'cmap': 'RdBu_r',
    },
    'I': {
        'y_label': r'$I\, [{\rm M_\odot\,yr^{-1}sr^{-1}}]$',
        'cmap': 'RdBu_r',
    },
}

In [ ]:
base_ecosys = ecosys_infos['z0/lm10']

def forward_r(x):
    x = np.asarray(x)
    return np.piecewise(x,
                        [x < base_ecosys.R_ISM, (x >= base_ecosys.R_ISM) &
                         (x <= base_ecosys.R_CGM),
                         x > base_ecosys.R_CGM],
                        [lambda x: x * 50 / base_ecosys.R_ISM,
                         lambda x: (x - base_ecosys.R_ISM) * 50 /
                         (base_ecosys.R_CGM - base_ecosys.R_ISM) + 50,
                         lambda x: (x - base_ecosys.R_CGM) * 50 /
                         (2000 - base_ecosys.R_CGM) + 100])


def inverse_r(y):
    y = np.asarray(y)
    return np.piecewise(
        y, [y < 50, (y >= 50) & (y <= 100),
            y > 100],
        [lambda y: base_ecosys.R_ISM / 50 * y,
         lambda y: (base_ecosys.R_CGM - base_ecosys.R_ISM) / 50 * (y - 50) +
         base_ecosys.R_ISM, lambda y: (2000 - base_ecosys.R_CGM) / 50 *
         (y - 100) + base_ecosys.R_CGM])


theta_ticklabels = [
    r'$\theta=90^{\circ}$', r'$60^{\circ}$', r'$30^{\circ}$', r'$0^{\circ}$',
    r'$-30^{\circ}$', r'$-60^{\circ}$', r'$-90^{\circ}$', r'$-60^{\circ}$',
    r'$-30^{\circ}$', r'$0^{\circ}$', r'$30^{\circ}$', r'$60^{\circ}$']
theta_ticks = np.arange(0, 360, 30)
r_ticks=[20, 40, 200, 400, 1000, 2000]
arrow_prop = dict(arrowstyle='-|>', lw=1., fc='grey', ec='grey')

def conf_pi_plot(ax: plt.Axes,
                 R_h: float,
                 R_h_text_loc=1.5,
                 ax_loc=-38,
                 tk_ang=-40.,
                 ax_r_start=0.,
                 ax_r_end=2000*1.75,
                 tick_dL=4,
                 tick_white=None | str):

    ax.set_theta_zero_location('N')
    ax.set_theta_direction(1)
    ax.set_rscale('function', functions=(forward_r, inverse_r))
    for deg in np.radians(range(0, 360, 30)):
        ax.plot([deg, deg], [1800, 2000], color='k', lw=1., zorder=10)
    if tick_white is not None:
        if tick_white == 'l':
            tks = np.radians(range(0, 180, 30))
        else:
            tks = np.radians(range(180, 360, 30))
        for deg in tks:
            ax.plot([deg, deg], [1800, 2000], color='w', lw=1., zorder=11)

    ax.set_thetagrids(theta_ticks, labels=theta_ticklabels)
    ax.grid(False)
    ax.tick_params(pad=8.)

    ax_loc = np.deg2rad(ax_loc)
    ax.annotate('', xy=(ax_loc, ax_r_end), xytext=(ax_loc, ax_r_start), arrowprops=arrow_prop, annotation_clip=False)
    ax.text(ax_loc, ax_r_end * 1., r'$r\,[{\rm kpc}]$', fontsize=15, ha='left',va='center', zorder=3)
    for r in r_ticks:
        ax.plot([ax_loc, ax_loc+np.arctan(tick_dL/(base_ecosys.R_ISM/50*forward_r(r)))],
                [r, np.sqrt(r*r+tick_dL*tick_dL)], c='grey', lw=1.)

    ax.set_rgrids(r_ticks, angle=tk_ang, fontsize=9, ha='left', va='center')    
    
    kw = dict(c='grey',linestyle=(0,(1,2)),linewidth=1)
    _ts = np.arange(0,2*np.pi,0.001)
    _ones = np.ones_like(_ts)
    ax.plot(_ts,base_ecosys.R_ISM*_ones,**kw)
    ax.plot(_ts,base_ecosys.R_CGM*_ones,**kw)
    
    ax.text(np.deg2rad(-135), R_h*R_h_text_loc, r'$R_{\rm h}$', va='center', ha='center', fontsize=15, zorder=100, c='k')
    ax.plot(np.linspace(1.1*np.pi,1.4*np.pi,20),[R_h]*20,c='grey',lw=1)

In [ ]:
fig, axs = plot.subplots(3, share=False, space=0., subsize=(5.0, 5.0), 
                         margin=[0.1, 0.05, 0.205, 0.05], layout='none')
axs_f = axs.flat
for ax in axs:
    ax.axis_off()
    
maps = fields_2D['z0/lm10']
ecosys = ecosys_infos['z0/lm10']

fig = fig._raw 
axs = [ax._raw for ax in axs_f]
    
plots=[(2,'v_phi',(-150.,150), 'v_r',(-125,125)),
       (3,'lambda',(-0.15,0.15), 'I',(-.8,.8)),
       (1,'lg_n',(-6.,0), 'lg_T',(4,6)),]

for i_ax, (axk, key1, lim1, key2, lim2) in enumerate(plots):
    ax=plt.subplot(1,3,axk,projection='polar')
    
    d1 = maps[key1]['median']
    cmap1, label1 = y_confs[key1]['cmap'], y_confs[key1]['y_label']
    d2 = maps[key2]['median']
    cmap2, label2 = y_confs[key2]['cmap'], y_confs[key2]['y_label']
    
    conf_pi_plot(ax, R_h=ecosys.R_h, tick_white=('l' if i_ax == 2 else None))
    
    
    hb1=ax.pcolormesh(theta_ms, r_ms, d1.T,vmin=lim1[0],vmax=lim1[1],cmap=cmap1, rasterized=True)
    hb2=ax.pcolormesh(2*np.pi-theta_ms, r_ms, d2.T,vmin=lim2[0],vmax=lim2[1],cmap=cmap2, rasterized=True)
    cax1=inset_axes(ax,width="45%",height="4%",bbox_to_anchor=(-0.4,-0.5,1,1),bbox_transform=ax.transAxes,
                    loc='lower left',borderpad=6)
    cbar1=fig.colorbar(hb1,cax=cax1,orientation='horizontal')
    cax1.set_xlabel(label1, labelpad=0)
    cax2=inset_axes(ax,width="45%",height="4%",bbox_to_anchor=(0.4,-0.5,1,1),bbox_transform=ax.transAxes,
                    loc='lower right',borderpad=6)
    cbar2=fig.colorbar(hb2,cax=cax2,orientation='horizontal')
    cax2.set_xlabel(label2, labelpad=0)
    
fig.savefig(out_fig_dir/'mass10z0_imgs.pdf')

In [ ]:
fig, axs = plt.subplots(3,3,subplot_kw={'projection':'polar'},figsize=(12.5, 13.5), gridspec_kw={
    'hspace': .1, 'wspace': .2, 'left': 0.03, 'right': 0.97, 'top': 0.97, 'bottom': 0.08
})

maps_list = fields_2D['z0']['lm9', 'lm10', 'lm11']
ecosys_list = ecosys_infos['z0']['lm9', 'lm10', 'lm11']
reduc_key = 'mean'
# reduc_key = 'median'

plots=[(0,0, 0, 'lg_n', (-6,0), 'lg_T', (4,6)),
       (1,0, 1, 'lg_n', (-6,0), 'lg_T', (4,6)),
       (2,0, 2, 'lg_n', (-6,0), 'lg_T', (4,6)),
       (0,1, 0, 'v_phi', (-150,150), 'v_r', (-150,150)),
       (1,1, 1, 'v_phi', (-150,150), 'v_r', (-150,150)),
       (2,1, 2, 'v_phi', (-150,150), 'v_r', (-150,150)),
       (0,2, 0, 'lambda', (-0.2,0.2), 'I', (-1,1)),
       (1,2, 1, 'lambda', (-0.2,0.2), 'I', (-1,1)),
       (2,2, 2, 'lambda', (-0.2,0.2), 'I', (-1,1))]

arrow_prop = dict(arrowstyle='-|>',lw=1., fc='grey', ec='grey')
r_h_scales = [1.55, 1.4, 1.3]
for rowid, colid, map_id, key1, lim1, key2, lim2 in plots:
    ax = axs[rowid, colid]    
    
    maps, ecosys = maps_list[map_id], ecosys_list[map_id]
    R_h = ecosys.R_h
    d1 = maps[key1][reduc_key]
    cmap1, label1 = y_confs[key1]['cmap'], y_confs[key1]['y_label']
    d2 = maps[key2][reduc_key]
    cmap2, label2 = y_confs[key2]['cmap'], y_confs[key2]['y_label']
    
    conf_pi_plot(ax, R_h=R_h, R_h_text_loc=r_h_scales[rowid],
                 tick_white=('l' if colid==0 else None))
    
    
    hb1=ax.pcolormesh(theta_ms, r_ms,d1.T,vmin=lim1[0],vmax=lim1[1],cmap=cmap1, rasterized=True)
    hb2=ax.pcolormesh(2*np.pi-theta_ms, r_ms,d2.T,vmin=lim2[0],vmax=lim2[1],cmap=cmap2, rasterized=True)
    
    if rowid == 2:
        cax1=inset_axes(ax,width="45%",height="4%",bbox_to_anchor=(-.365,-0.5,1,1),bbox_transform=ax.transAxes,loc='lower left',borderpad=6)
        cbar1=fig.colorbar(hb1,cax=cax1,orientation='horizontal')
        cbar1.set_label(label1)
        cax2=inset_axes(ax,width="45%",height="4%",bbox_to_anchor=(.365,-0.5,1,1),bbox_transform=ax.transAxes,loc='lower right',borderpad=6)
        cbar2=fig.colorbar(hb2,cax=cax2,orientation='horizontal')
        cbar2.set_label(label2)

# fig.savefig(out_fig_dir/'mdependence1_.pdf')
fig.savefig(out_fig_dir/f'mdependence1_{reduc_key}.pdf')

In [ ]:
fig, axs = plt.subplots(3,3,subplot_kw={'projection':'polar'},figsize=(12.5, 13.5), gridspec_kw={
    'hspace': .1, 'wspace': .2, 'left': 0.03, 'right': 0.97, 'top': 0.97, 'bottom': 0.08
})

maps_list = fields_2D['z2/lm10', 'z1/lm10', 'z0/lm10']
ecosys_list = ecosys_infos['z2/lm10', 'z1/lm10', 'z0/lm10']

plots=[(0,0, 0, 'lg_n', (-6,0), 'lg_T', (4,6)),
       (1,0, 1, 'lg_n', (-6,0), 'lg_T', (4,6)),
       (2,0, 2, 'lg_n', (-6,0), 'lg_T', (4,6)),
       (0,1, 0, 'v_phi', (-150,150), 'v_r', (-150,150)),
       (1,1, 1, 'v_phi', (-150,150), 'v_r', (-150,150)),
       (2,1, 2, 'v_phi', (-150,150), 'v_r', (-150,150)),
       (0,2, 0, 'lambda', (-0.2,0.2), 'I', (-1,1)),
       (1,2, 1, 'lambda', (-0.2,0.2), 'I', (-1,1)),
       (2,2, 2, 'lambda', (-0.2,0.2), 'I', (-1,1))]
arrow_prop = dict(arrowstyle='-|>',lw=1., fc='grey', ec='grey')
r_h_scales = [1.75, 1.6, 1.45]
for rowid, colid, map_id, key1, lim1, key2, lim2 in plots:
    ax = axs[rowid, colid]    
    
    maps, ecosys = maps_list[map_id], ecosys_list[map_id]
    R_h = ecosys.R_h
    d1 = maps[key1]['median']
    cmap1, label1 = y_confs[key1]['cmap'], y_confs[key1]['y_label']
    d2 = maps[key2]['median']
    cmap2, label2 = y_confs[key2]['cmap'], y_confs[key2]['y_label']
    
    conf_pi_plot(ax, R_h=R_h, R_h_text_loc=r_h_scales[rowid],
                 tick_white=('l' if colid==0 else None))
    
    hb1=ax.pcolormesh(theta_ms,r_ms,d1.T,vmin=lim1[0],vmax=lim1[1],cmap=cmap1, rasterized=True)
    hb2=ax.pcolormesh(2*np.pi-theta_ms,r_ms,d2.T,vmin=lim2[0],vmax=lim2[1],cmap=cmap2, rasterized=True)
    
    if rowid == 2:
        cax1=inset_axes(ax,width="45%",height="4%",bbox_to_anchor=(-.365,-0.5,1,1),bbox_transform=ax.transAxes,loc='lower left',borderpad=6)
        cbar1=fig.colorbar(hb1,cax=cax1,orientation='horizontal')
        cbar1.set_label(label1)
        cax2=inset_axes(ax,width="45%",height="4%",bbox_to_anchor=(.365,-0.5,1,1),bbox_transform=ax.transAxes,loc='lower right',borderpad=6)
        cbar2=fig.colorbar(hb2,cax=cax2,orientation='horizontal')
        cbar2.set_label(label2)

fig.savefig(out_fig_dir/'zdependence1_.pdf')

In [ ]:
fig, axs = plot.subplots(3, share=False, space=0., subsize=(5.0, 5.0), 
                         margin=[0.1, 0.05, 0.205, 0.05], layout='none')
axs_f = axs.flat
for ax in axs:
    ax.axis_off()
    
fig = fig._raw 
axs = [ax._raw for ax in axs_f]

maps = fields_2D['z0/lm11_hiBH']
ecosys = ecosys_infos['z0/lm11_hiBH']
    
plots=[(2, 'v_phi', (-150.,150), 'v_r', (-150,150)),
       (3, 'lambda', (-0.2,0.2), 'I', (-1.,1.)),
       (1, 'lg_n', (-6.,0), 'lg_T', (4,6))]

for i_ax, (axk,key1,lim1,key2,lim2) in enumerate(plots):
    ax=plt.subplot(1,3,axk,projection='polar')
    
    d1 = maps[key1]['median']
    cmap1, label1 = y_confs[key1]['cmap'], y_confs[key1]['y_label']
    d2 = maps[key2]['median']
    cmap2, label2 = y_confs[key2]['cmap'], y_confs[key2]['y_label']
    
    conf_pi_plot(ax, R_h=ecosys.R_h, R_h_text_loc=0.75, tick_white=('l' if i_ax == 2 else None))
    
    hb1=ax.pcolormesh(theta_ms,r_ms,d1.T,vmin=lim1[0],vmax=lim1[1],cmap=cmap1, rasterized=True)
    hb2=ax.pcolormesh(2*np.pi-theta_ms,r_ms,d2.T,vmin=lim2[0],vmax=lim2[1],cmap=cmap2, rasterized=True)
    cax1=inset_axes(ax,width="45%",height="4%",bbox_to_anchor=(-0.4,-0.5,1,1),bbox_transform=ax.transAxes,
                    loc='lower left',borderpad=6)
    cbar1=fig.colorbar(hb1,cax=cax1,orientation='horizontal')
    cax1.set_xlabel(label1, labelpad=0)
    cax2=inset_axes(ax,width="45%",height="4%",bbox_to_anchor=(0.4,-0.5,1,1),bbox_transform=ax.transAxes,
                    loc='lower right',borderpad=6)
    cbar2=fig.colorbar(hb2,cax=cax2,orientation='horizontal')
    cax2.set_xlabel(label2, labelpad=0)
    
fig.savefig(out_fig_dir/'highmassSMBH_imgs.pdf')

## Field computation

In [ ]:
%autoreload
from baryon_cycle.utils import FieldInterpolatorND

def interp_field(dsets: dict[str, np.ndarray],
                 header: dict[str, np.ndarray]):
    r_es, phi_es, theta_es = header['r_edges', 'phi_edges', 'theta_edges']
    theta_es = np.pi/2. - theta_es
    r_cs, phi_cs, theta_cs = .5 * (r_es[:-1] + r_es[1:]), \
        .5 * (phi_es[:-1] + phi_es[1:]), \
        .5 * (theta_es[:-1] + theta_es[1:])
    phi_ms, r_ms, theta_ms = np.meshgrid(phi_cs, r_cs, theta_cs, indexing='ij')
    phis, rs, thetas = phi_ms.ravel(), r_ms.ravel(), theta_ms.ravel()

    n_phis = len(phi_cs)
    keys = tuple(dsets.keys())
    n_gals = len(dsets[keys[0]]) // n_phis
    print(f'Working on {n_gals=}, {keys=}')

    dsets_out = {k: v.copy() for k, v in dsets.items()}
    for i_gal in range(n_gals):
        b = i_gal * n_phis
        e = b + n_phis
        interps = {}
        for key in keys:
            field = dsets[key][b:e]
            vals = field.ravel()
            filled = ~np.isnan(vals) 
            interp_key = 'dm' if key.startswith('DM_') else 'b'
            if interp_key not in interps:
                interps[interp_key] = FieldInterpolatorND.from_astro_spherical(
                    rs, thetas, phis, filled)
            interp = interps[interp_key]
            
            vals_o = interp.eval(vals)
            field_o = vals_o.reshape(field.shape)
            n_nan = np.isnan(field_o).sum()
            
            assert n_nan == 0, f'{key} has nans in output ({n_nan} nans)'
            dsets_out[key][b:e] = field_o

    return dsets_out

In [ ]:
f_in = sim_dir / '2Dmaps_individual.hdf5'
fields_2D = h5.File.load_from(f_in)

In [ ]:
header = fields_2D['header']
fields_2D_interp = {'header': header}

keys_list = [
    ('z0', 'lm9'),
    ('z0', 'lm10'),
    ('z0', 'lm11'),
    ('z0', 'lm11_hiBH'),
    ('z1', 'lm10'),
    ('z2', 'lm10'),
]
for z_key, m_key in keys_list:
    out = interp_field(fields_2D[z_key][m_key], header)
    fields_2D_interp.setdefault(z_key, {})[m_key] = out

In [ ]:
f_out = sim_dir / '2Dmaps_individual_interp.hdf5'
h5.File.dump_to(f_out, fields_2D_interp, f_flag='w')

In [ ]:
h5.File.ls_from(f_out)

In [ ]:
## Check interpolation vs original

d_raw, d_new = fields_2D['z0']['lm10'], fields_2D_interp['z0']['lm10']

fig, axs = plot.subplots((2,2), share=False, space=0.25, subsize=(5.0, 2.5), margin=[0.1, 0.1, 0.1, 0.1], layout='none')
axs_f = axs.flat

f0 = d_raw['lg_T'][:24]
f1 = d_new['lg_T'][:24]
ax = axs_f[0]
ax._raw.imshow(np.nanmedian(f0, 0).T)

ax = axs_f[1]
ax._raw.imshow(np.median(f1, 0).T)

f0 = d_raw['lambda_z'][:24]
f1 = d_new['lambda_z'][:24]
ax = axs_f[2]
ax._raw.imshow(np.log10(np.nanmedian(f0, 0)).T)

ax = axs_f[3]
ax._raw.imshow(np.log10(np.median(f1, 0)).T)

In [ ]:
f_in = sim_dir / '2Dmaps_individual_interp.hdf5'
fields_2D = h5.File.load_from(f_in)

In [ ]:
header = fields_2D['header'].copy()
header.pop('phi_edges')

In [ ]:
keys_list = [
    ('z0', 'lm9'),
    ('z0', 'lm10'),
    ('z0', 'lm11'),
    ('z0', 'lm11_hiBH'),
    ('z1', 'lm10'),
    ('z2', 'lm10'),
]
out = {'header': header}
for z_key, m_key in keys_list:
    dsets = fields_2D[z_key][m_key]
    print(z_key, m_key)
    for k, v in dsets.items():
        print(' - ', k, v.shape)
        median = np.median(v, axis=0)
        mean = np.mean(v, axis=0)
        stddev = np.std(v, axis=0)
        out.setdefault(z_key, {}).setdefault(m_key, {})[k] = {
            'median': median,
            'mean': mean,
            'stddev': stddev,
        }

In [ ]:
f_out = sim_dir/ '2Dmaps_interp.hdf5'
h5.File.dump_to(f_out, out, f_flag='w')

In [ ]:
h5.File.ls_from(f_out)

# 3D fields

In [ ]:
from baryon_cycle.utils import FieldInterpolator2DCartesian

## Interp and stack

In [ ]:
fields_3D = h5.File.load_from(sim_dir / '3Dmaps_individual.hdf5')

In [ ]:
header = fields_3D['header']

In [ ]:
fields_3D_interp = {
    'header': header,
}
for scale_key in 'ISM', 'CGM', 'IGM':
    xbins, ybins = header[f'bin_edges/{scale_key}']['x', 'y']
    dx, dy = xbins[1] - xbins[0], ybins[1] - ybins[0]
    x_cs, y_cs = xbins[:-1] + .5 * dx, ybins[:-1] + dy

    for view_key in 'edge_on', 'face_on':
        OHs =fields_3D[f'{view_key}/{scale_key}']['OH']
        print(scale_key, view_key, OHs.shape, np.isnan(OHs).sum())
        OHs_interp = np.empty_like(OHs)
        for gid, OH in enumerate(OHs):
            interp = FieldInterpolator2DCartesian(x_cs, y_cs, ~np.isnan(OH))
            OHs_interp[gid] = interp.eval(OH)
            
        fields_3D_interp.setdefault(view_key, {}).setdefault(scale_key, {})['OH'] = OHs_interp

In [ ]:
h5.File.dump_to(sim_dir / '3Dmaps_individual_interp.hdf5', fields_3D_interp, f_flag='w')

In [ ]:
fields_3D_interp = {}
for scale_key in 'ISM', 'CGM', 'IGM':
    xbins, ybins = header[f'bin_edges/{scale_key}']['x', 'y']
    dx, dy = xbins[1] - xbins[0], ybins[1] - ybins[0]
    x_cs, y_cs = xbins[:-1] + .5 * dx, ybins[:-1] + dy
    for view_key in 'edge_on', 'face_on':
        for prop_key in 'v_x', 'v_y':
            vals =fields_3D[f'{view_key}/{scale_key}/{prop_key}']
            print(scale_key, view_key, prop_key, vals.shape, np.isnan(vals).sum())
            vals_interp = np.empty_like(vals)
            for gid, val in enumerate(vals):
                interp = FieldInterpolator2DCartesian(x_cs, y_cs, ~np.isnan(val))
                vals_interp[gid] = interp.eval(val)
            fields_3D_interp.setdefault(view_key, {}).setdefault(scale_key, {})[prop_key] = vals_interp

In [ ]:
h5.File.dump_to(sim_dir / '3Dmaps_individual_interp.hdf5', fields_3D_interp, 
                f_flag='a', g_flag='ac', dump_flag='ac')

In [ ]:
h5.File.ls_from(sim_dir / '3Dmaps_individual_interp.hdf5')

In [ ]:
fields_gals = h5.File.load_from(sim_dir / '3Dmaps_individual_interp.hdf5')

In [ ]:
fields_stacked = {}
for k_view in 'edge_on', 'face_on':
    fields_stacked[k_view] = {}
    for k_scale in 'ISM', 'CGM', 'IGM':
        fields_stacked[k_view][k_scale] = {}
        for prop_key in 'v_x', 'v_y', 'OH':
            vals = fields_gals[k_view][k_scale][prop_key]
            print(k_view, k_scale, prop_key, vals.shape)
            median = np.median(vals, axis=0)
            mean = np.mean(vals, axis=0)
            stddev = np.std(vals, axis=0)
            fields_stacked[k_view][k_scale][prop_key] = {
                'median': median,
                'mean': mean,
                'stddev': stddev,
            }
fields_stacked['header'] = fields_gals['header']

In [ ]:
h5.File.dump_to(sim_dir / '3Dmaps_interp.hdf5', fields_stacked, f_flag='x')

## Stacked - velocity

In [ ]:
def conf_ax(ax: plot.Axes, lim=50,
            sbar_size=10, sbar_pad=10, vsize=1, loc='lower right'):
    
    ax.lim([-lim, lim], [-lim, lim])
    
    rax = ax._raw
    rax.set_aspect('equal')
    rax.grid(False)
    rax.set_facecolor('k')
    rax.xaxis.set_ticks([])
    rax.yaxis.set_ticks([])
    rax.tick_params(axis='both', color='w', width=1, length=0)
    
    scalebar = AnchoredSizeBar(rax.transData,
                           sbar_size, '', loc, 
                           pad=0,
                           color='white',
                           frameon=False,
                           size_vertical=vsize, borderpad=sbar_pad, 
                           fontproperties={'size': 0})
    rax.add_artist(scalebar)

In [ ]:
fields = h5.File.load_from(sim_dir / '3Dmaps_interp.hdf5')

In [ ]:
norm_f = mcolors.Normalize(vmin=0,vmax=120)
norm_e = mcolors.Normalize(vmin=0,vmax=65)

norm_div = mcolors.Normalize(vmin=-65,vmax=65)

thetas = np.linspace(0,2*np.pi,1000)
R_h = ecosys_infos['z0/lm10'].R_h

In [ ]:
fig, ax = plot.subplots(1, figsize=(2.25, .85), margin=[0.05, 0.05, 0.785, 0.02], layout='none')

sm = mpl.cm.ScalarMappable(cmap='rainbow', norm=norm_f)
fig.colorbar(sm, cax=ax._raw, 
             location='bottom', label=r'$v\,[{\rm km/s}]$')

plot.savefig(out_fig_dir/'streamline_fcb.pdf')

In [ ]:
fig, ax = plot.subplots(1, figsize=(2.25, .85), margin=[0.05, 0.05, 0.785, 0.02], layout='none')

sm = mpl.cm.ScalarMappable(cmap='rainbow', norm=norm_e)
fig.colorbar(sm, cax=ax._raw, 
             location='bottom', label=r'$v\,[{\rm km/s}]$')

plot.savefig(out_fig_dir/'streamline_ecb.pdf')

In [ ]:
fig, ax = plot.subplots(1, figsize=(2.25, .85), margin=[0.05, 0.05, 0.785, 0.02], layout='none')

sm = mpl.cm.ScalarMappable(cmap='bwr', norm=norm_div)
fig.colorbar(sm, cax=ax._raw, 
             location='bottom', label=r'$v\,[{\rm km/s}]$')

plot.savefig(out_fig_dir/'streamline_divcb.pdf')

In [ ]:
x_es, y_es = fields['header']['bin_edges/ISM']['x', 'y']
dx, dy = x_es[1] - x_es[0], y_es[1] - y_es[0]
x_cs, y_cs = x_es[:-1] + .5 * dx, y_es[:-1] + dy 

v_x, v_y = fields['face_on/ISM']['v_x/median', 'v_y/median']
v_norm=np.hypot(v_x, v_y)

In [ ]:
fig, ax = plot.subplots(1, figsize=6., margin=[0.01, 0.01, 0.01, 0.01], layout='none')

strm= ax._raw.streamplot(x_cs, y_cs, v_x.T, v_y.T, color=v_norm.T, 
    cmap='rainbow',norm=norm_f,linewidth=1.,
    density=3,#3,
    arrowsize=.8)

conf_ax(ax, 39, 10, 10, 1)
plot.savefig(out_fig_dir/'streamline_f3.pdf')

In [ ]:
v_x, v_y = fields['edge_on/ISM']['v_x/median', 'v_y/median']
v_norm=np.hypot(v_x, v_y)

In [ ]:
fig, ax = plot.subplots(1, figsize=6., margin=[0.01, 0.01, 0.01, 0.01], layout='none')

strm= ax._raw.streamplot(x_cs, y_cs, v_x.T, v_y.T, color=v_norm.T,
    cmap='rainbow',norm=norm_e,linewidth=1.,density=3,arrowsize=.8)

conf_ax(ax, 39, 10, 10, 1)
plot.savefig(out_fig_dir/'streamline_e3.pdf')

In [ ]:
x_es, y_es = fields['header']['bin_edges/CGM']['x', 'y']
dx, dy = x_es[1] - x_es[0], y_es[1] - y_es[0]
x_cs, y_cs = x_es[:-1] + .5 * dx, y_es[:-1] + dy 

v_x, v_y = fields['face_on/CGM']['v_x/median', 'v_y/median']
v_norm=np.hypot(v_x, v_y)

In [ ]:
fig, ax = plot.subplots(1, figsize=6., margin=[0.01, 0.01, 0.01, 0.01], layout='none')

strm= ax._raw.streamplot(x_cs, y_cs, v_x.T, v_y.T, color=v_norm.T, 
    cmap='rainbow',norm=norm_f, linewidth=1.,density=5, #5, 
    arrowsize=.8)
ax.plot(R_h*np.cos(thetas),R_h*np.sin(thetas),c='grey',ls=(0,(2,1)),lw=2.0)

conf_ax(ax, 199, 50, 10, 5, loc='lower left')
plot.savefig(out_fig_dir/'streamline_f2.pdf')

In [ ]:
v_x, v_y = fields['edge_on/CGM']['v_x/median', 'v_y/median']
v_norm=np.hypot(v_x, v_y)

In [ ]:
fig, ax = plot.subplots(1, figsize=6., margin=[0.01, 0.01, 0.01, 0.01], layout='none')

strm= ax._raw.streamplot(x_cs, y_cs, v_x.T, v_y.T, color=v_norm.T, 
    cmap='rainbow',norm=norm_e,linewidth=1.,density=5,arrowsize=.8)
ax.plot(R_h*np.cos(thetas),R_h*np.sin(thetas),c='grey',ls=(0,(2,1)),lw=2.0)

conf_ax(ax, 199, 50, 10, 5,'lower left')

plot.savefig(out_fig_dir/'streamline_e2.pdf')

In [ ]:
x_es, y_es = fields['header']['bin_edges/IGM']['x', 'y']
dx, dy = x_es[1] - x_es[0], y_es[1] - y_es[0]
x_cs, y_cs = x_es[:-1] + .5 * dx, y_es[:-1] + dy 

v_x, v_y = fields['face_on/IGM']['v_x/median', 'v_y/median']
v_norm=np.hypot(v_x, v_y)

In [ ]:
fig, ax = plot.subplots(1, figsize=12.5, margin=[0.01, 0.01, 0.01, 0.01], layout='none')

strm=ax._raw.streamplot(x_cs, y_cs, v_x.T, v_y.T, color=v_norm.T, 
    cmap='rainbow',norm=norm_f,linewidth=1., density=15,#15, 
    arrowsize=.8)

x, y = R_h*np.cos(thetas),R_h*np.sin(thetas)
ax.plot(x,y,c='grey',ls=(0,(2,1)),lw=2.0)

n_x=len(x)
b,e=13*n_x//16, 15*n_x//16
ax.plot(2.*x[b:e],2.*y[b:e],c='grey',ls=(0,(2,1)),lw=2.0)

conf_ax(ax, 675, 200, 20, 8)

plot.savefig(out_fig_dir/'streamline_f1.pdf')

In [ ]:
fig, ax = plot.subplots(1, figsize=12.5, margin=[0.01, 0.01, 0.01, 0.01], layout='none')

strm=ax._raw.streamplot(x_cs, y_cs, v_x.T, v_y.T, color=v_norm.T, 
    cmap='rainbow',norm=norm_f,linewidth=1., density=15,#15, 
    arrowsize=.8)

conf_ax(ax, 675, 200, 20, 8)

plot.savefig(out_fig_dir/'streamline_f1-nomarkup.pdf')

In [ ]:
x_ms, y_ms = np.meshgrid(x_cs, y_cs)
dot_prod = x_ms * v_x.T + y_ms * v_y.T
v_norm_signed = np.sign(dot_prod) * v_norm.T

In [ ]:
fig, ax = plot.subplots(1, figsize=12.5, margin=[0.01,]*4, layout='none')

strm=ax._raw.streamplot(x_cs, y_cs, v_x.T, v_y.T, color=v_norm_signed, 
    cmap='bwr',norm=norm_div,
    linewidth=1.,density=15, arrowsize=.8
)
x, y = R_h*np.cos(thetas),R_h*np.sin(thetas)
ax.plot(R_h*np.cos(thetas),R_h*np.sin(thetas),c='grey',ls=(0,(2,1)),lw=2.0)

n_x=len(x)
b,e=13*n_x//16, 15*n_x//16
ax.plot(2.*x[b:e],2.*y[b:e],c='grey',ls=(0,(2,1)),lw=2.0)

conf_ax(ax, 675, 200, 20, 8)

In [ ]:
v_x, v_y = fields['edge_on/IGM']['v_x/median', 'v_y/median']
v_norm=np.hypot(v_x, v_y)

In [ ]:
fig, ax = plot.subplots(1, figsize=12.5, margin=[0.01,]*4, layout='none')

strm=ax._raw.streamplot(x_cs, y_cs, v_x.T, v_y.T, color=v_norm.T, 
    cmap='rainbow',norm=norm_e,linewidth=1.,density=15, arrowsize=.8
)
x, y = R_h*np.cos(thetas),R_h*np.sin(thetas)
ax.plot(R_h*np.cos(thetas),R_h*np.sin(thetas),c='grey',ls=(0,(2,1)),lw=2.0)

n_x=len(x)
b,e=13*n_x//16, 15*n_x//16
ax.plot(2.*x[b:e],2.*y[b:e],c='grey',ls=(0,(2,1)),lw=2.0)

conf_ax(ax, 675, 200, 20, 8)
plot.savefig(out_fig_dir/'streamline_e1.pdf')

In [ ]:
x_ms, y_ms = np.meshgrid(x_cs, y_cs)
dot_prod = x_ms * v_x.T + y_ms * v_y.T
v_norm_signed = np.sign(dot_prod) * v_norm.T

In [ ]:
fig, ax = plot.subplots(1, figsize=12.5, margin=[0.01,]*4, layout='none')

strm=ax._raw.streamplot(x_cs, y_cs, v_x.T, v_y.T, color=v_norm_signed, 
    cmap='bwr',norm=norm_div,
    linewidth=1.,density=15, arrowsize=.8
)
x, y = R_h*np.cos(thetas),R_h*np.sin(thetas)
ax.plot(R_h*np.cos(thetas),R_h*np.sin(thetas),c='grey',ls=(0,(2,1)),lw=2.0)

n_x=len(x)
b,e=13*n_x//16, 15*n_x//16
ax.plot(2.*x[b:e],2.*y[b:e],c='grey',ls=(0,(2,1)),lw=2.0)

conf_ax(ax, 675, 200, 20, 8)

## Stacked - metal

In [ ]:
fields_3D = h5.File.load_from(sim_dir / '3Dmaps_individual_interp.hdf5')

In [ ]:
norm= mcolors.Normalize(vmin=7., vmax=9.)
norm2 = mcolors.Normalize(vmin=7., vmax=8.)

In [ ]:
fig, axs = plot.subplots((2,3), share=False, space=(0.125, 0.105), subsize=(5.5, 5.5), 
                         margin=[0.02, 0.02, 0.125, 0.1], layout='none')
axs_f = axs.flat

lims = 675., 199., 39.
strm_dens = 3, 1.75, 1.75
strm_cs = 'khaki', 'k', 'k'
for i_view, view_key in enumerate(('face_on', 'edge_on')):
    for i_scale, scale_key in enumerate(('IGM', 'CGM', 'ISM')):
        ax = axs[i_view, i_scale]
        xbins, ybins = fields_3D[f'header/bin_edges/{scale_key}']['x', 'y']
        dx, dy = xbins[1] - xbins[0], ybins[1] - ybins[0]
        x_cs, y_cs = xbins[:-1] + .5 * dx, ybins[:-1] + dy

        OHs, v_xs, v_ys =fields_3D[f'{view_key}/{scale_key}']['OH', 'v_x', 'v_y']
        
        OH_stack = np.median(OHs, axis=0)
        print(OHs.shape, np.isnan(OHs).sum())
        ax._raw.pcolormesh(x_cs, y_cs, OH_stack.T, 
                        cmap='jet', norm=norm, rasterized=True)

        v_x, v_y = np.median(v_xs, axis=0), np.median(v_ys, axis=0)
        strm=ax._raw.streamplot(x_cs, y_cs, v_x.T, v_y.T,
                        #color=strm_cs[i_scale],
                        color=OH_stack.T,
                        cmap='Greys',norm=norm2,
                        linewidth=.75, density=strm_dens[i_scale], 
                        arrowsize=.8, zorder=10)
        

        x_lim = y_lim = -lims[i_scale], lims[i_scale]
        ax.lim(x_lim, y_lim)

for ax in axs_f:
    ax._raw.set_aspect('equal')
    ax._raw.grid(False)
    ax._raw.set_facecolor('k')
    ax._raw.tick_params(axis='both', color='w', width=1, length=5)

axs[0,0].label(y=r'y\,[{\rm kpc}]')
axs[1,0].label(y=r'z\,[{\rm kpc}]')
axs[1].label(r'x\,[{\rm kpc}]')
    
plot.savefig(out_fig_dir/'metal-field.pdf')

In [ ]:
fig, ax = plot.subplots(1, figsize=(2.25, .85), margin=[0.05, 0.05, 0.785, 0.02], layout='none')

sm = mpl.cm.ScalarMappable(cmap='jet', norm=norm)
fig.colorbar(sm, cax=ax._raw,  location='bottom', label=r'$12+\log({\rm O/H})$')

plot.savefig(out_fig_dir/'metal-cb.pdf')

## Individuals - velocity

In [ ]:
fields_3D = h5.File.load_from(sim_dir / '3Dmaps_individual_interp.hdf5')

In [ ]:
xbins, ybins = fields_3D['header/bin_edges/CGM']['x', 'y']
dx, dy = xbins[1] - xbins[0], ybins[1] - ybins[0]
x_cs, y_cs = xbins[:-1] + .5 * dx, ybins[:-1] + dy

v_x, v_y =fields_3D['face_on/CGM']['v_x', 'v_y']
v_x.shape, np.isnan(v_x).sum()

In [ ]:
gid = 0
interp = FieldInterpolator2DCartesian(x_cs, y_cs, ~np.isnan(v_x[gid]))
v_xo = interp.eval(v_x[gid])
v_yo = interp.eval(v_y[gid])

In [ ]:
rng = Rng(10086)
gids = rng.choice(131, 4)
gids
# => array([130,  68, 107,  78])

In [ ]:
fig, axs = plot.subplots((2,4), share=False, space=0.02, subsize=(4.25, 4.25), 
                         margin=[0.02, 0.02, 0.1, 0.1], layout='none')
axs_f = axs.flat

wxfs, wyfs = fields_3D['face_on/CGM']['v_x', 'v_y']
wxes, wyes = fields_3D['edge_on/CGM']['v_x', 'v_y']

xbins, ybins = fields_3D['header/bin_edges/CGM']['x', 'y']
dx, dy = xbins[1] - xbins[0], ybins[1] - ybins[0]
x_cs, y_cs = xbins[:-1] + .5 * dx, ybins[:-1] + dy

normf=mcolors.Normalize(vmin=0,vmax=120)
norme=mcolors.Normalize(vmin=0,vmax=60)
kw = dict(cmap='rainbow',linewidth=.5, density=8, arrowsize=.8, broken_streamlines=True)

gids = [130,  68, 107,  78]

for i_g, gid in enumerate(gids):
    fields_3D
    wxf, wyf = wxfs[gid], wyfs[gid]
    wxe, wye = wxes[gid], wyes[gid]
    
    # interp = FieldInterpolator2DCartesian(x_cs, y_cs, ~np.isnan(wxf))
    # wxf, wyf = interp.eval(wxf), interp.eval(wyf)
    # interp = FieldInterpolator2DCartesian(x_cs, y_cs, ~np.isnan(wxe))
    # wxe, wye = interp.eval(wxe), interp.eval(wye)

    speedf=np.hypot(wxf,wyf)
    speede=np.hypot(wxe,wye)
    
    ax = axs[0,i_g]
    strm= ax._raw.streamplot(x_cs, y_cs, wxf.T, wyf.T, color=speedf,norm=normf,**kw)
    
    ax = axs[1,i_g]
    strm= ax._raw.streamplot(x_cs, y_cs, wxe.T, wye.T, color=speede,norm=norme,**kw)
    
for ax in axs_f:
    ax.lim([-125., 125.], [-125., 125.]).label(r'x\,[{\rm kpc}]', r'y\,[{\rm kpc}]').label_outer()
    ax._raw.set_aspect('equal')
    ax._raw.grid(False)
    ax._raw.set_facecolor('k')
    ax._raw.tick_params(axis='both', color='w', width=1, length=5)
axs[1,0].label(y=r'z\,[{\rm kpc}]')

# plot.savefig(out_fig_dir/'streamline-individuals.pdf')

In [ ]:
fig, axs = plot.subplots((2,4), share=False, space=0.02, subsize=(4.25, 4.25), 
                         margin=[0.02, 0.02, 0.1, 0.1], layout='none')
axs_f = axs.flat

wxfs, wyfs = fields_3D['face_on/CGM']['v_x', 'v_y']
wxes, wyes = fields_3D['edge_on/CGM']['v_x', 'v_y']

xbins, ybins = fields_3D['header/bin_edges/CGM']['x', 'y']
dx, dy = xbins[1] - xbins[0], ybins[1] - ybins[0]
x_cs, y_cs = xbins[:-1] + .5 * dx, ybins[:-1] + dy

normf=mcolors.Normalize(vmin=0,vmax=120)
norme=mcolors.Normalize(vmin=0,vmax=60)
kw = dict(cmap='rainbow',linewidth=.5, density=8, arrowsize=.8, broken_streamlines=True)

gids = [0+0, 0+8, 0+15, 0+17]

for i_g, gid in enumerate(gids):
    fields_3D
    wxf, wyf = wxfs[gid], wyfs[gid]
    wxe, wye = wxes[gid], wyes[gid]
    
    # interp = FieldInterpolator2DCartesian(x_cs, y_cs, ~np.isnan(wxf))
    # wxf, wyf = interp.eval(wxf), interp.eval(wyf)
    # 
    # interp = FieldInterpolator2DCartesian(x_cs, y_cs, ~np.isnan(wxe))
    # wxe, wye = interp.eval(wxe), interp.eval(wye)

    speedf=np.hypot(wxf,wyf)
    speede=np.hypot(wxe,wye)
    
    ax = axs[0,i_g]
    strm= ax._raw.streamplot(x_cs, y_cs, wxf.T, wyf.T, color=speedf,norm=normf,**kw)
    
    ax = axs[1,i_g]
    strm= ax._raw.streamplot(x_cs, y_cs, wxe.T, wye.T, color=speede,norm=norme,**kw)
    
for ax in axs_f:
    ax.lim([-125., 125.], [-125., 125.]).label(r'x\,[{\rm kpc}]', r'y\,[{\rm kpc}]').label_outer()
    ax._raw.set_aspect('equal')
    ax._raw.grid(False)
    ax._raw.set_facecolor('k')
    ax._raw.tick_params(axis='both', color='w', width=1, length=5)
axs[1,0].label(y=r'z\,[{\rm kpc}]')

plot.savefig(out_fig_dir/'streamline-individuals-selected.pdf')

In [ ]:
fig, axs = plot.subplots((5,4), share=False, space=0.02, subsize=(4.25, 4.25), 
                         margin=[0.02, 0.02, 0.1, 0.1], layout='none')
axs_f = axs.flat

wxfs, wyfs = fields_3D['face_on/CGM']['v_x', 'v_y']
wxes, wyes = fields_3D['edge_on/CGM']['v_x', 'v_y']

xbins, ybins = fields_3D['header/bin_edges/CGM']['x', 'y']
dx, dy = xbins[1] - xbins[0], ybins[1] - ybins[0]
x_cs, y_cs = xbins[:-1] + .5 * dx, ybins[:-1] + dy

normf=mcolors.Normalize(vmin=0,vmax=120)
norme=mcolors.Normalize(vmin=0,vmax=60)
kw = dict(cmap='rainbow',linewidth=.5, density=4, arrowsize=.8, broken_streamlines=True)

rng = Rng(10086)
gids = np.arange(0,0+20)

for i_g, gid in enumerate(gids):
    fields_3D
    wxf, wyf = wxfs[gid], wyfs[gid]
    wxe, wye = wxes[gid], wyes[gid]
    
    #interp = FieldInterpolator2DCartesian(x_cs, y_cs, ~np.isnan(wxe))
    #wxe, wye = interp.eval(wxe), interp.eval(wye)
    speede=np.hypot(wxe,wye)
    
    ax = axs_f[i_g]
    strm= ax._raw.streamplot(x_cs, y_cs, wxe.T, wye.T, color=speede,norm=norme,**kw)
    
for ax in axs_f:
    ax.lim([-125., 125.], [-125., 125.]).label(r'x\,[{\rm kpc}]', r'y\,[{\rm kpc}]').label_outer()
    ax._raw.set_aspect('equal')
    ax._raw.grid(False)
    ax._raw.set_facecolor('k')
    ax._raw.tick_params(axis='both', color='w', width=1, length=5)

In [ ]:
fig, axs = plot.subplots((5,4), share=False, space=0.02, subsize=(4.25, 4.25), 
                         margin=[0.02, 0.02, 0.1, 0.1], layout='none')
axs_f = axs.flat

wxfs, wyfs = fields_3D['face_on/CGM']['v_x', 'v_y']
wxes, wyes = fields_3D['edge_on/CGM']['v_x', 'v_y']

xbins, ybins = fields_3D['header/bin_edges/CGM']['x', 'y']
dx, dy = xbins[1] - xbins[0], ybins[1] - ybins[0]
x_cs, y_cs = xbins[:-1] + .5 * dx, ybins[:-1] + dy

normf=mcolors.Normalize(vmin=0,vmax=120)
norme=mcolors.Normalize(vmin=0,vmax=60)
kw = dict(cmap='rainbow',linewidth=.5, density=4, arrowsize=.8, broken_streamlines=True)

rng = Rng(10086)
gids = np.arange(0,0+20)

for i_g, gid in enumerate(gids):
    fields_3D
    
    wxf, wyf = wxfs[gid], wyfs[gid]
    speedf=np.hypot(wxf,wyf)
    
    ax = axs_f[i_g]
    strm= ax._raw.streamplot(x_cs, y_cs, wxf.T, wyf.T, color=speedf, norm=normf, **kw)
    
for ax in axs_f:
    ax.lim([-125., 125.], [-125., 125.]).label(r'x\,[{\rm kpc}]', r'y\,[{\rm kpc}]').label_outer()
    ax._raw.set_aspect('equal')
    ax._raw.grid(False)
    ax._raw.set_facecolor('k')
    ax._raw.tick_params(axis='both', color='w', width=1, length=5)

In [ ]:
fig, axs = plot.subplots((5,4), share=False, space=0.02, subsize=(4.25, 4.25), 
                         margin=[0.02, 0.02, 0.1, 0.1], layout='none')
axs_f = axs.flat

wxfs, wyfs = fields_3D['face_on/ISM']['v_x', 'v_y']
wxes, wyes = fields_3D['edge_on/ISM']['v_x', 'v_y']

xbins, ybins = fields_3D['header/bin_edges/ISM']['x', 'y']
dx, dy = xbins[1] - xbins[0], ybins[1] - ybins[0]
x_cs, y_cs = xbins[:-1] + .5 * dx, ybins[:-1] + dy

normf=mcolors.Normalize(vmin=0,vmax=120)
norme=mcolors.Normalize(vmin=0,vmax=60)
kw = dict(cmap='rainbow',linewidth=.5, density=4, arrowsize=.8, broken_streamlines=True)

rng = Rng(10086)
gids = np.arange(0,0+20)

for i_g, gid in enumerate(gids):
    fields_3D
    
    wxf, wyf = wxfs[gid], wyfs[gid]
    speedf=np.hypot(wxf,wyf)
    
    ax = axs_f[i_g]
    strm= ax._raw.streamplot(x_cs, y_cs, wxf.T, wyf.T, color=speedf, norm=normf, **kw)
    
for ax in axs_f:
    ax.lim([-40., 40.], [-40., 40.]).label(r'x\,[{\rm kpc}]', r'y\,[{\rm kpc}]').label_outer()
    ax._raw.set_aspect('equal')
    ax._raw.grid(False)
    ax._raw.set_facecolor('k')
    ax._raw.tick_params(axis='both', color='w', width=1, length=5)

In [ ]:
fig, axs = plot.subplots((5,4), share=False, space=0.02, subsize=(4.25, 4.25), 
                         margin=[0.02, 0.02, 0.1, 0.1], layout='none')
axs_f = axs.flat

wxfs, wyfs = fields_3D['face_on/IGM']['v_x', 'v_y']
wxes, wyes = fields_3D['edge_on/IGM']['v_x', 'v_y']

xbins, ybins = fields_3D['header/bin_edges/IGM']['x', 'y']
dx, dy = xbins[1] - xbins[0], ybins[1] - ybins[0]
x_cs, y_cs = xbins[:-1] + .5 * dx, ybins[:-1] + dy

normf=mcolors.Normalize(vmin=0,vmax=120)
norme=mcolors.Normalize(vmin=0,vmax=60)
kw = dict(cmap='rainbow',linewidth=.5, density=10, arrowsize=.8, broken_streamlines=True)

rng = Rng(10086)
gids = np.arange(0,0+20)

for i_g, gid in enumerate(gids[:10]):
    fields_3D
    
    wxf, wyf = wxfs[gid], wyfs[gid]
    speedf=np.hypot(wxf,wyf)
    
    ax = axs_f[i_g]
    strm= ax._raw.streamplot(x_cs, y_cs, wxf.T, wyf.T, color=speedf, norm=normf, **kw)
    
for ax in axs_f:
    ax.lim([-500., 500.], [-500., 500.]).label(r'x\,[{\rm kpc}]', r'y\,[{\rm kpc}]').label_outer()
    ax._raw.set_aspect('equal')
    ax._raw.grid(False)
    ax._raw.set_facecolor('k')
    ax._raw.tick_params(axis='both', color='w', width=1, length=5)

# The stacking method

## Individual vs stacked

In [ ]:
fields_1D_inds = h5.File.load_from(sim_dir / '1Dprofiles_individual_interp.hdf5')
fields_1D = h5.File.load_from(sim_dir / '1Dprofiles_interp.hdf5')

header = fields_1D_inds['header']
theta_edges, r_edges = header['theta_edges', 'r_edges']
r_cs = .5 * (r_edges[:-1] + r_edges[1:])

In [ ]:
samp_key = 'z0/lm10'
ecosys = ecosys_infos[samp_key]

data = {}
for i_prop, prop_key in enumerate(('v_phi', 'v_r')):
    for i_direc, direc_key in enumerate(('disk', 'polar')):
        
        v_med = fields_1D[samp_key][f'{prop_key}/{direc_key}/median']
        v_inds = fields_1D_inds[samp_key][f'{prop_key}/{direc_key}'][:,:,0]
        v_inds = v_inds.reshape(24,-1,len(r_cs))
        print(v_inds.shape)
        #q_lo, q_hi = np.quantile(v_inds, (.16, .84), axis=(0,1))
        q_lo, q_hi = np.quantile(np.median(v_inds, axis=0), (.16, .84), axis=0)
        dv = .5 * (q_hi - q_lo)
        ratio = dv / np.abs(v_med)
        
        data[(i_prop, i_direc)] = ratio

In [ ]:
fig, axs = plot.subplots((1, 2), share=False, 
                         space=(.3, .035), 
                         subsize=(5*.9, 2.5*.9), 
                         margin=[0.02, 0.05, 0.12, 0.05], 
                         layout='none')
axs_f = axs.flat

y_lims = [0.02, 165.], [0.02, 185.]
x_nodes = [1., ecosys.R_ISM, ecosys.R_CGM, 2200]
regime_alphas = .75, .5, .25
regime_cs = cs_named['blue', 'green','orange']

y_labs= r'v_\phi\, [{\rm km/s}]', r'v_r\, [{\rm km/s}]'
direc_labs = r'$\text{Disk plane}$', r'$\text{Outflow cone}$'
kws = dict(lw=3.5), dict(lw=1)
for i_prop in range(2):
    ax = axs[i_prop]
    for i_direc in range(2):
        val = data[i_prop, i_direc]
        ax.plot(r_cs, val, label=direc_labs[i_direc], **kws[i_direc])
        
    for k in range(3):
        y_lb, y_ub = y_lims[i_prop] 
        y_ub = y_lb * 10.
        ax.fill_between([x_nodes[k], x_nodes[k+1]], [y_lb]*2, [y_ub]*2, 
                        color=regime_cs[k], alpha=regime_alphas[k], 
                        lw=0)
        
    ax.lim([1., 500.], y_lims[i_prop] )\
        .scale('log', 'log')\
        .label(r'r\, [{\rm kpc}]', y_labs[i_prop])

In [ ]:
fields_3D = h5.File.load_from(sim_dir / '3Dmaps_interp.hdf5')
fields_3D_inds = h5.File.load_from(sim_dir / '3Dmaps_individual_interp.hdf5')

In [ ]:
xbins, ybins = fields_3D['header/bin_edges/CGM']['x', 'y']
dx, dy = xbins[1] - xbins[0], ybins[1] - ybins[0]
x_cs, y_cs = xbins[:-1] + .5 * dx, ybins[:-1] + dy

In [ ]:
data = {}

for i_view, view in enumerate(('face_on', 'edge_on')):
    v_x_inds, v_y_inds = fields_3D_inds[f'{view}/CGM']['v_x', 'v_y']
    v_norm_inds = np.hypot(v_x_inds, v_y_inds)
    v_norm_lo, v_norm_hi = np.quantile(v_norm_inds, (.16, .84), axis=0)
    dv_norm = (v_norm_hi - v_norm_lo) * .5
    
    v_x, v_y = fields_3D[f'{view}/CGM']['v_x/median', 'v_y/median']
    v_norm = np.hypot(v_x, v_y)
    
    ratio = dv_norm / v_norm
    
    data[i_view] = ratio

In [ ]:
fig, axs = plot.subplots((1,2), share=False, space=0.02, subsize=(4.5, 4.5), 
                         margin=[0.02, 0.02, 0.125, 0.1], layout='none')
axs_f = axs.flat

norms = [mcolors.Normalize(vmin=0., vmax=vmax) for vmax in (2., 2.)]

for i_ax in range(2):
    ax = axs_f[i_ax]
    ratio = data[i_ax]
    lt = ax._raw.pcolormesh(xbins, ybins, ratio.T, cmap='coolwarm', 
                       norm=norms[i_ax], rasterized=True)

for ax in axs_f:
    ax.lim([-200., 200.], [-200., 200.]).label(r'x\,[{\rm kpc}]', r'z\,[{\rm kpc}]')    
    ax._raw.set_aspect('equal')
    ax._raw.grid(False)
    ax._raw.set_facecolor('k')
    ax._raw.tick_params(axis='both', color='w', width=1, length=5)
    axs.label_outer()
    
fig.colorbar(lt, ax=axs_f, location='right', 
             label=r'$\sigma_v / v$', aspect=25, pad=.02, fraction=.015)
    
# plot.savefig(out_fig_dir/'model-edge-on.pdf')

In [ ]:
data = {}
for i_view, view in enumerate(('face_on', 'edge_on')):
    f_inds = fields_3D_inds[f'{view}/CGM']['OH']
    f_lo, f_hi = np.quantile(f_inds, (.16, .84), axis=0)
    df = (f_hi - f_lo) * .5    
    data[i_view] = df

In [ ]:
fig, axs = plot.subplots((1,2), share=False, space=0.02, subsize=(4.5, 4.5), 
                         margin=[0.02, 0.02, 0.125, 0.1], layout='none')
axs_f = axs.flat

norms = [mcolors.Normalize(vmin=0., vmax=vmax) for vmax in (.3, .3)]

for i_ax in range(2):
    ax = axs_f[i_ax]
    ratio = data[i_ax]
    lt = ax._raw.pcolormesh(xbins, ybins, ratio.T, cmap='coolwarm', 
                       norm=norms[i_ax], rasterized=True)

for ax in axs_f:
    ax.lim([-200., 200.], [-200., 200.]).label(r'x\,[{\rm kpc}]', r'z\,[{\rm kpc}]')    
    ax._raw.set_aspect('equal')
    ax._raw.grid(False)
    ax._raw.set_facecolor('k')
    ax._raw.tick_params(axis='both', color='w', width=1, length=5)
    axs.label_outer()
    
fig.colorbar(lt, ax=axs_f, location='right', 
             label=r'$\sigma_\text{[O/H]}$', aspect=25, pad=.02, fraction=.015)
    
# plot.savefig(out_fig_dir/'model-edge-on.pdf')

## Mean vs Median

In [ ]:
def conf_ax(ax: plot.Axes, lim=50,
            sbar_size=10, sbar_pad=10, vsize=1, loc='lower right'):
    
    ax.lim([-lim, lim], [-lim, lim])
    
    rax = ax._raw
    rax.set_aspect('equal')
    rax.grid(False)
    rax.set_facecolor('k')
    rax.xaxis.set_ticks([])
    rax.yaxis.set_ticks([])
    rax.tick_params(axis='both', color='w', width=1, length=0)
    
    scalebar = AnchoredSizeBar(rax.transData,
                           sbar_size, '', loc, 
                           pad=0,
                           color='white',
                           frameon=False,
                           size_vertical=vsize, borderpad=sbar_pad, 
                           fontproperties={'size': 0})
    rax.add_artist(scalebar)

In [ ]:
fields = h5.File.load_from(sim_dir / '3Dmaps_interp.hdf5')

In [ ]:
norm_f = mcolors.Normalize(vmin=0,vmax=120)
norm_e = mcolors.Normalize(vmin=0,vmax=65)
thetas = np.linspace(0,2*np.pi,1000)
R_h = ecosys_infos['z0/lm10'].R_h

In [ ]:
x_es, y_es = fields['header']['bin_edges/IGM']['x', 'y']
dx, dy = x_es[1] - x_es[0], y_es[1] - y_es[0]
x_cs, y_cs = x_es[:-1] + .5 * dx, y_es[:-1] + dy

dset = fields['edge_on/IGM']
v_x1, v_y1 = dset['v_x/median', 'v_y/median']
v_x2, v_y2 = dset['v_x/mean', 'v_y/mean']
v_norm1 = np.hypot(v_x1, v_y1)
v_norm2 = np.hypot(v_x2, v_y2)

v_x, v_y = v_x2 - v_x1, v_y2 - v_y1
v_norm=np.hypot(v_x, v_y)

In [ ]:
fig, ax = plot.subplots(1, figsize=12.5, margin=[0.01,]*4, layout='none')

strm=ax._raw.streamplot(x_cs, y_cs, v_x.T, v_y.T, color=v_norm.T, 
    cmap='rainbow',norm=norm_e,linewidth=1.,density=15, arrowsize=.8
)
ax.plot(R_h*np.cos(thetas),R_h*np.sin(thetas),c='grey',ls=(0,(2,1)),lw=1.5)

conf_ax(ax, 675, 200, 20, 0)

In [ ]:
fig, ax = plot.subplots(1, figsize=12.5, margin=[0.01,]*4, layout='none')

strm=ax._raw.streamplot(x_cs, y_cs, v_x2.T, v_y2.T, color=v_norm2.T, 
    cmap='rainbow',norm=norm_e,linewidth=1.,density=15, arrowsize=.8
)
ax.plot(R_h*np.cos(thetas),R_h*np.sin(thetas),c='grey',ls=(0,(2,1)),lw=1.5)

conf_ax(ax, 675, 200, 20, 0)

In [ ]:
fig, ax = plot.subplots(1, figsize=12.5, margin=[0.01,]*4, layout='none')

strm=ax._raw.streamplot(x_cs, y_cs, v_x1.T, v_y1.T, color=v_norm1.T, 
    cmap='rainbow',norm=norm_e,linewidth=1.,density=15, arrowsize=.8
)
ax.plot(R_h*np.cos(thetas),R_h*np.sin(thetas),c='grey',ls=(0,(2,1)),lw=1.5)

conf_ax(ax, 675, 200, 20, 0)

In [ ]:
dset = fields['face_on/IGM']
v_x1, v_y1 = dset['v_x/median', 'v_y/median']
v_x2, v_y2 = dset['v_x/mean', 'v_y/mean']
v_norm1 = np.hypot(v_x1, v_y1)
v_norm2 = np.hypot(v_x2, v_y2)

v_x, v_y = v_x2 - v_x1, v_y2 - v_y1
v_norm=np.hypot(v_x, v_y)

In [ ]:
fig, ax = plot.subplots(1, figsize=12.5, margin=[0.01, 0.01, 0.01, 0.01], layout='none')

strm=ax._raw.streamplot(x_cs, y_cs, v_x.T, v_y.T, color=v_norm.T, 
    cmap='rainbow',norm=norm_f,linewidth=1., density=15,#15, 
    arrowsize=.8)
ax.plot(R_h*np.cos(thetas),R_h*np.sin(thetas),c='grey',ls=(0,(2,1)),lw=1.5)

conf_ax(ax, 675, 200, 20, 0)

In [ ]:
fig, ax = plot.subplots(1, figsize=12.5, margin=[0.01, 0.01, 0.01, 0.01], layout='none')

strm=ax._raw.streamplot(x_cs, y_cs, v_x2.T, v_y2.T, color=v_norm2.T, 
    cmap='rainbow',norm=norm_f,linewidth=1., density=15,#15, 
    arrowsize=.8)
ax.plot(R_h*np.cos(thetas),R_h*np.sin(thetas),c='grey',ls=(0,(2,1)),lw=1.5)

conf_ax(ax, 675, 200, 20, 0)

In [ ]:
fig, ax = plot.subplots(1, figsize=12.5, margin=[0.01, 0.01, 0.01, 0.01], layout='none')

strm=ax._raw.streamplot(x_cs, y_cs, v_x1.T, v_y1.T, color=v_norm1.T, 
    cmap='rainbow',norm=norm_f,linewidth=1., density=15,#15, 
    arrowsize=.8)
ax.plot(R_h*np.cos(thetas),R_h*np.sin(thetas),c='grey',ls=(0,(2,1)),lw=1.5)

conf_ax(ax, 675, 200, 20, 0)

# The theory

## Spin

In [ ]:
f_in = sim_dir / '1Dprofiles_interp.hdf5'
fields_1D = h5.File.load_from(f_in)

header = fields_1D['header']
theta_edges, r_edges = header['theta_edges', 'r_edges']

inc_edges = -theta_edges + np.pi/2
absinc_edges = inc_edges[25:]
absinc_cs = .5 * (absinc_edges[:-1] + absinc_edges[1:])

absinc_cs[-1] = np.pi/2.
absinc_cs[0] = 0.

r_cs = .5 * (r_edges[:-1] + r_edges[1:])

In [ ]:
profs = fields_1D['z0/lm10']
gas_lx = profs['lambda_x/disk/median']
gas_ly = profs['lambda_y/disk/median']
gas_lz = profs['lambda_z/disk/median']
gas_ltot = np.sqrt(gas_lx**2 + gas_ly**2 + gas_lz**2)

DM_lx = profs['DM_lambda_x/disk/median']
DM_ly = profs['DM_lambda_y/disk/median']
DM_lz = profs['DM_lambda_z/disk/median']
DM_l_tot = np.sqrt(DM_lx**2 + DM_ly**2 + DM_lz**2)

In [ ]:
fig, axs = plot.subplots((2,1), share=(True,False), 
                         space=0.025, 
                         subsize=(6.0, 3.25), 
                         margin=[0.025, 0.025, 0.085, 0.15], layout='none')
axs_f = axs.flat

ax = axs_f[0]
ax.plot(r_cs, gas_lz, lw=3.,label=r'$\left|\vec{\lambda}\right|$')
ax.plot(r_cs, gas_ltot, lw=3.,label=r'$\left|\vec{\lambda}\right|$')
#ax.fill_between(r_cs,w6moduleperc1,w6moduleperc2,alpha=0.5,color='grey')
ax.plot(r_cs, DM_lz, lw=3., c=cs_named['b'], label=r'$\lambda_z$')
#ax.fill_between(r_cs,w6perc1,w6perc2,alpha=0.3,color=cs_named['b'])

ax = axs_f[1]
#ax.plot(r_cs,w6cos,lw=3)
#ax.fill_between(r_cs,w6cosperc1,w6cosperc2,alpha=0.5,color='grey')

axs_f[0].label(y=r'\text{Spin}')\
    .scale('log','log')\
    .lim([1., 2.0e3], [10**-3.05, 10**1.25])\
    .leg(loc='ul', handlelength=1.2, labelcolor='linecolor')
axs_f[1].label(r'r\,[\text{kpc}]',r'\cos\,\left<\vec{\lambda},z\right>')\
    .lim(y=[-0.65, 1.25])
pan_labs = 'a', 'b'    
for i_ax, ax in enumerate(axs_f):
    ax.text(r'\,$%s$\,'%(pan_labs[i_ax]), (.94, .9), **panel_number_kw, ha='center')
#plot.savefig(out_fig_dir/'spinalignment.pdf')

## Dark matter vs baryon

In [ ]:
f_in = sim_dir / '1Dprofiles_interp.hdf5'
fields_1D = h5.File.load_from(f_in)

header = fields_1D['header']
theta_edges, r_edges = header['theta_edges', 'r_edges']

inc_edges = -theta_edges + np.pi/2
absinc_edges = inc_edges[25:]
absinc_cs = .5 * (absinc_edges[:-1] + absinc_edges[1:])

absinc_cs[-1] = np.pi/2.
absinc_cs[0] = 0.

r_cs = .5 * (r_edges[:-1] + r_edges[1:])

In [ ]:
ecosys = ecosys_infos['z0/lm10']
profs = fields_1D['z0/lm10']

In [ ]:
x = r_cs 

prof = profs['DM_lambda_z']
red_keys = 'median', 'q16', 'q84'

ys_dm = Num.safe_lg(prof['disk'][red_keys]) 

prof = profs['lambda']
ys_b = Num.safe_lg(prof['disk'][red_keys])

In [ ]:
fig, ax = plot.subplots(1, figsize=(5.5, 4.5), margin=[0.1, 0.1, 0.1, 0.1], layout='none')

cs = cs_named['k', 'b']
labs = r'$\rm{Dark\ matter}$', r'$\rm{Baryons}$'
for i_y, (y, y_lo, y_hi) in enumerate((ys_dm, ys_b)):
    ax.c(cs[i_y]).plot(x, y, lw=2.5, label=labs[i_y])\
        .fill_between(x, y_lo, y_hi, alpha=0.25)

ax.scale('log').lim([1., 2000.], [-2.75, -.5])\
    .label(r'r\, [{\rm kpc}]', r'\log\lambda')\
    .leg(loc='ul')

In [ ]:
f_in = sim_dir / '2Dmaps_individual_interp.hdf5'
fields_2D = h5.File.load_from(f_in)

In [ ]:
y_dm = np.mean(fields_2D['z0/lm10']['DM_v_r'], axis=(0,2)) 

In [ ]:
DM_v_r = np.median(fields_2D['z0/lm10']['DM_v_r'], axis=0)
B_v_r = np.median(fields_2D['z0/lm10']['v_r'], axis=0)

In [ ]:
fig, axs = plot.subplots((1,2), share=True, space=0.075, subsize=(6.5, 4.5), 
                         margin=[0.1, 0.1, 0.1, 0.1], layout='none')
axs_f = axs.flat

inc_deg_edges = np.rad2deg(inc_edges)

kw = dict(cmap='jet', vmin=-100, vmax=100)

for i_Z, Z in enumerate((B_v_r, DM_v_r)): 
    ax = axs_f[i_Z]
    art = ax._raw.pcolormesh(r_edges, inc_deg_edges, Z.T, **kw)

    ax.label(r'r\,[{\rm kpc}]', r'\theta\, [{\rm deg}]')
    
axs.label_outer()
axs_f[0].text(r'\rm Gas', (.92, .92), ha='right', c='w', fontsize=17)
axs_f[1].text(r'\rm Dark\ matter', (.92, .92), ha='right', c='w', fontsize=17)

fig.colorbar(art, ax=axs_f, pad=.02, label=r'$v_r\, [{\rm km/s}]$')

In [ ]:
fig, ax = plot.subplots(1, figsize=4.5, margin=[0.1, 0.1, 0.1, 0.1], layout='none')

ax.plot(x, y_dm)
ax.scale('log').lim([1., 2500.], [-25., 125.])

In [ ]:
fig, ax = plot.subplots(1, figsize=(6.25, 5.), margin=[0.1, 0.1, 0.1, 0.1], layout='none')

lab_direc = r'$(\text{Disk plane})$', r'$(\text{Outflow cone})$'
p_kw = dict(lw=2.5), dict(lw=1.5, ls=(0,(2,2)))

cs = cs_named['k', 'b']
labs = r'$\rm{Dark\ matter}$', r'$\rm{Baryons}$'
x = r_cs 

for i_direc, direc in enumerate(('disk', 'polar')):
    
    for i_y, y_key in enumerate(('DM_lambda_z', 'lambda')):
        prof = profs[y_key]
        y = prof[direc]['median']
        ax.c(cs[i_y]).plot(x, y, **p_kw[i_direc],
                           label=labs[i_y]+lab_direc[i_direc])#.fill_between(x, y_lo, y_hi, alpha=0.25)

ax.scale('log', 'log').lim([1., 2000.], [1.0e-5, 10.])\
    .label(r'r\, [{\rm kpc}]', r'\lambda')\
    .leg(loc='ul')

In [ ]:
fig, ax = plot.subplots(1, figsize=4.5, margin=[0.1, 0.1, 0.1, 0.1], layout='none')

ax.plot(x, y, c='k', lw=2.5)

## Outflow cone

In [ ]:
from scipy.signal import savgol_filter

class OutflowGeometry:
    def __init__(self, prop_maps: DataDict[str, np.ndarray], header: DataDict):
        
        v_r =  prop_maps['v_r/median'].T
        theta_edges, r_edges = header['theta_edges', 'r_edges']
        inc_edges = -theta_edges + np.pi/2
        s_inc_edges = np.sin(inc_edges)
        
        s_inc_cs = .5 * (s_inc_edges[:-1] + s_inc_edges[1:])
        r_cs = .5 * (r_edges[:-1] + r_edges[1:])
        # theta_2D, r_2D = np.meshgrid(theta_edges,r_edges,indexing='ij')
        
        self.v_r = v_r 
        self.theta_edges = theta_edges
        self.r_edges = r_edges
        self.inc_edges = inc_edges
        self.s_inc_edges = s_inc_edges
        self.s_inc_cs = s_inc_cs 
        self.r_cs = r_cs
        self.psi_outs_fwhm = self._find_psi_outs()
        
    @staticmethod
    def _find_thres(xs, ys, y_dst):
        inds = np.nonzero(ys > y_dst)[0] 
        if len(inds) == 0:
            i_r = 0
            #print(f"Warning: threshold not reached: {y_dst=}, {ys=}")
        else:
            i_r = inds[0]
        i_l = max(i_r - 1, 0)
        if i_l == i_r:
            x_dst = xs[i_l]
        else:
            x_l, x_r = xs[i_l], xs[i_r]
            y_l, y_r = ys[i_l], ys[i_r]
            k = (x_r - x_l) / (y_r - y_l)
            x_dst = x_l + k * (y_dst - y_l)
                    
        return x_dst

    @staticmethod
    def _find_psi_out(s_incs, v_rs, frac_v=.5):
        v_rs = savgol_filter(v_rs, len(v_rs)//5, 3)
        
        sel = s_incs > 0.
        x, y = s_incs[sel], v_rs[sel]
        n_y = len(y)
        y_dst = frac_v * y[-n_y//2:].max()
        x_hi = OutflowGeometry._find_thres(x, y, y_dst)
        inc_hi = np.arcsin(x_hi)
        
        sel = s_incs <= 0.
        x, y = s_incs[sel][::-1], v_rs[sel][::-1]
        n_y = len(y)
        y_dst = frac_v * y[-n_y//2:].max()
        x_lo = OutflowGeometry._find_thres(x, y, y_dst)
        inc_lo = np.arcsin(x_lo)
        
        dinc = inc_hi - inc_lo
        psi_out = np.pi - dinc
        return np.rad2deg(psi_out)
        
    def _find_psi_outs(self, frac_v = .5):
        s_incs, v_r_map = self.s_inc_cs, self.v_r
        psi_outs = [self._find_psi_out(s_incs, v_rs, frac_v=frac_v) 
                    for v_rs in v_r_map.T]
        hasout = [(v_rs.max() > 0.) 
                 #&(v_rs.min() < 0.) 
                 for v_rs in v_r_map.T]
        return np.array(psi_outs), np.array(hasout)

In [ ]:
fields_2D = h5.File.load_from(sim_dir / '2Dmaps_interp.hdf5')

geos = {
    key: OutflowGeometry(fields_2D[key], fields_2D['header'])
    #'z0_lm10': OutflowGeometry(all_data['z0/lm10'], all_data['header']),
    #'z0_lm11': OutflowGeometry(all_data['z0/lm11'], all_data['header']),
    #'z0_lm11_hiBH': OutflowGeometry(all_data['z0/lm11_hiBH'], all_data['header']),
    #'z1_lm10': OutflowGeometry(all_data['z1/lm10'], all_data['header']),
    #'z2_lm10': OutflowGeometry(all_data['z2/lm10'], all_data['header'])
    for key in ['z0/lm9', 'z0/lm10', 'z0/lm11', 'z0/lm11_hiBH', 'z1/lm10', 'z2/lm10']
}

In [ ]:
geo = geos['z0/lm11']

In [ ]:
fig, ax = plot.subplots(1, figsize=4.5, margin=[0.1, 0.1, 0.1, 0.1], layout='none')

rs, incs, v_r_map = geo.r_edges, np.rad2deg(geo.inc_edges), geo.v_r

x = .5 * (incs[:-1] + incs[1:])
y = v_r_map[:, len(rs)//2]
y = savgol_filter(y, len(y)//5, 3)

sel = x > 0.
x_dst = OutflowGeometry._find_thres(x[sel], y[sel], y[sel].max()*.5)
ax.plot([x_dst, x_dst], [-55., 55.], 'k--')
sel = x < 0.
x_dst = OutflowGeometry._find_thres(x[sel][::-1], y[sel][::-1], y[sel].max()*.5)
ax.plot([x_dst, x_dst], [-55., 55.], 'k--')

ax.plot(x, y)
ax.lim([-90., 90.])

In [ ]:
fig, ax = plot.subplots(1, figsize=4.5, margin=[0.1, 0.1, 0.1, 0.1], layout='none')

rs, incs, v_r_map = geo.r_edges, np.rad2deg(geo.inc_edges), geo.v_r
ax._raw.pcolormesh(rs, incs, v_r_map, cmap='RdBu_r', vmin=-55., vmax=55.)

r, psi, valid = geo.r_cs, 90.0 - geo.psi_outs_fwhm[0]/2., geo.psi_outs_fwhm[1]
ax.plot(r[valid], psi[valid], c='k', lw=2)

ax.lim([0., 400.])

In [ ]:
fig, axs = plot.subplots((1,2), share=(False, True), space=.025, 
                         subsize=(5.75, 5.5), margin=[0.02, 0.02, 0.09, 0.065], layout='none')
axs_f = axs.flat

ax = axs_f[0]

m_cs = cs_named['b', 'g', 'orange']
y_lb, y_ub = y_lim = [-10., 189.]
dy = (y_ub - y_lb)
for i_lm, lm in enumerate(['9', '10', '11']):
    m_c = m_cs[i_lm]
    
    geo = geos[f'z0/lm{lm}']
    rs, (psis, hasout) = geo.r_cs, geo.psi_outs_fwhm
    only_in = (~hasout)&(rs>25.)
    ax.c(m_c)
    ax.plot(rs, psis, lw=3.)
    ax.plot(rs[only_in], psis[only_in], lw=3.*1.25, c='w', ls=(0,(1.5,1.5)))

    
    r_ism, r_cgm = ecosys_defs[f'z0/lm{lm}']['R_ISM', 'R_CGM']
    ax.fill_between([r_ism, r_cgm], [y_lb+dy*(.25-i_lm*.05)]*2, [y_lb+dy*(.20-i_lm*.05)]*2, 
                    color=m_c, alpha=.75, lw=0)
    #arrow_prop = {'arrowstyle': '-|>','lw': 1., 'fc': m_c, 'ec': m_c}
    #ax._raw.annotate('', (r_ism, y_lb+dy*.15), xytext=(r_ism, y_lb+dy*.075), 
    #                    arrowprops=arrow_prop|{'lw': 1.})
    #ax._raw.annotate('', (r_cgm, y_lb+dy*.15), xytext=(r_cgm, y_lb+dy*.075), 
    #                    arrowprops=arrow_prop|{'lw': 3.5})

m_c = "#904313"
i_lm = 3
lm = '11_hiBH'

geo = geos[f'z0/lm{lm}']
rs, (psis, hasout) = geo.r_cs, geo.psi_outs_fwhm
only_in = (~hasout)&(rs>25.)
ax.c(m_c)
ax.plot(rs, psis, lw=3.)
ax.plot(rs[only_in], psis[only_in], lw=3.*1.25, c='w', ls=(0,(1.5,1.5)))
    
r_ism, r_cgm = ecosys_defs[f'z0/lm{lm}']['R_ISM', 'R_CGM']
ax.fill_between([r_ism, r_cgm], [y_lb+dy*(.25-i_lm*.05)]*2, [y_lb+dy*(.20-i_lm*.05)]*2, 
                color=m_c, alpha=.75, lw=0)
    
ax = axs_f[1]
i_lm, lm = 1, '10'
z_cs = cs_named['g'], "#9cdb66", "#cbb162"
z_lws = 3., 2.5, 1.25
for i_z, z in enumerate(['0', '1', '2']):
    z_c = z_cs[i_z]
    z_lw = z_lws[i_z]
    
    geo = geos[f'z{z}/lm{lm}']
    rs, (psis, hasout) = geo.r_cs, geo.psi_outs_fwhm
    only_in = (~hasout)&(rs>25.)
    ax.c(z_c)
    ax.plot(rs, psis, lw=z_lw)
    ax.plot(rs[only_in], psis[only_in], lw=z_lw*1.25, c='w', ls=(0,(1.5,1.5)))

    r_ism, r_cgm = ecosys_defs[f'z{z}/lm{lm}']['R_ISM', 'R_CGM']
    ax.fill_between([r_ism, r_cgm], [y_lb+dy*(.2-i_z*.05)]*2, [y_lb+dy*(.15-i_z*.05)]*2, 
                    color=z_c, alpha=.75, lw=0)
    #arrow_prop = {'arrowstyle': '-|>','lw': 1., 'fc': z_c, 'ec': z_c}
    #ax._raw.annotate('', (r_ism, y_lb+dy*.15), xytext=(r_ism, y_lb+dy*.075), 
    #                    arrowprops=arrow_prop|{'lw': 1.})
    #ax._raw.annotate('', (r_cgm, y_lb+dy*.15), xytext=(r_cgm, y_lb+dy*.075), 
    #                    arrowprops=arrow_prop|{'lw': 3.5})

axs_f[0].lim([-15., 675.], y_lim)\
    .label(r'r\,[{\rm kpc}]', r'\psi_{\rm out}\,[{\rm deg}]')
axs_f[1].lim([-15., 675.], y_lim)\
    .label(r'r\,[{\rm kpc}]')

fig._raw.savefig(out_fig_dir/'outflow-geometry.pdf')

## Scaling with halo mass

In [ ]:
#all_data = h5.File.load_from(sim_dir / 'radial_profiles.hdf5')
fields_1D = h5.File.load_from(sim_dir / '1Dprofiles_interp.hdf5')

In [ ]:
class HaloScaling:
    def __init__(self, profs: DataDict[str, np.ndarray],
                 header: DataDict, ecosys: EcosysInfo):
        r_edges = header['r_edges']
        r_cs = .5 * (r_edges[1:] + r_edges[:-1])

        self.sim_info = ecosys.sim_info
        self.profs = profs
        self.header = header
        self.ecosys = ecosys

        self.r_edges = r_edges
        self.r_cs = r_cs

        self.derived = DataDict()

        self.__derive_props()

    def __derive_props(self):
        ecosys, profs = self.ecosys, self.profs
        M_h, R_h, R_ISM, R_CGM = (
            ecosys.M_h, ecosys.R_h, ecosys.R_ISM, ecosys.R_CGM)

        r_c = self.r_cs
        v_phi_max = profs['v_phi/disk/median'].max()
        r_nodes = {
            'ISM_1o5': R_ISM/5.,
            'CGM_1o5': R_CGM/5.,
            'ISM_half': R_ISM/2.,
            'CGM_half': R_CGM/2.,
            'h': R_h,
        }
        props = ['lg_T', 'v_phi', 'lg_n', 'v_r', 'I', 'lambda']
        for prop in props:
            for r_node, r in r_nodes.items():
                ys_major, ys_minor = self.profs[prop][
                    'disk/median', 'polar/median']
                y_major = np.interp(r, r_c, ys_major)
                y_minor = np.interp(r, r_c, ys_minor)
                self.derived |= {
                    f'{prop}_major_at_{r_node}': y_major,
                    f'{prop}_minor_at_{r_node}': y_minor
                }

        self.derived |= {
            'M_h': M_h,
            'R_h': R_h,
            'R_ISM': R_ISM,
            'R_CGM': R_CGM,
            'v_phi_max': v_phi_max,
        }


hscales = {
    'z0': {
        lm_key: HaloScaling(fields_1D[f'z0/{lm_key}'],
                            fields_1D['header'],
                            ecosys_infos[f'z0/{lm_key}'])
        for lm_key in ['lm9', 'lm10', 'lm11']
    },
}

In [ ]:
def draw_scaling(x_range, gamma=1., y0 = 1.):
    xs = np.linspace(*x_range, 10)
    ys = (xs / xs[0])**gamma * y0
    return xs, ys   

In [ ]:
plt_data = DataDict({})
for i_m, m_val in enumerate(['lm9', 'lm10', 'lm11']):
    hscale = hscales['z0'][m_val]
    keys = hscale.derived.keys() 
    for key in keys:
        plt_data.setdefault(key, []).append(hscale.derived[key])
for key in plt_data.keys():
    plt_data[key] = np.array(plt_data[key])

In [ ]:
fig, axs = plot.subplots((2,3), share=(True, False), space=(0.31, .065), 
                         subsize=(3.75, 3.), margin=[0.02, 0.02, 0.09, 0.075], layout='none')
axs_f = axs.flat
plt_kws = {
    'ISM': dict(s=50, marker='o'),
    'CGM': dict(s=35, marker='s'),
}
axis_kw = {
    'major': dict(c=cs_named['b'], fa=.3, elw=2),
    'minor': dict(c=cs_named['b'], fa=.3, elw=2),
}
x = plt_data['M_h']
x_tick_lim = [10**11.25, 10**11.75]
ax = axs_f[0]
y = 10.0**plt_data['lg_n_major_at_ISM_1o5']
ax.fmt_marker(**axis_kw['major']).scatter(x, y, **plt_kws['ISM'])
y = 10.0**plt_data['lg_n_minor_at_ISM_1o5']
ax.fmt_marker(c=cs_named['r'], fa=.3, elw=2).scatter(x, y, **plt_kws['ISM'])

y = 10.0**plt_data['lg_n_major_at_CGM_half']
ax.fmt_marker(**axis_kw['major']).scatter(x, y, **plt_kws['CGM'])
y = 10.0**plt_data['lg_n_minor_at_CGM_half']
ax.fmt_marker(c=cs_named['r'], fa=.3, elw=2).scatter(x, y, **plt_kws['CGM'])

x1, y1 = draw_scaling(x_tick_lim, y0=2.0e-2, gamma=1./3.)
ax.plot(x1, y1, lw=1.5, ls=(0,(2,2)), c='gray')
ax.lim(y=[1.0e-6, 1.0]).label(y=r'n\, [{\rm cm}^{-3}]')

ax = axs_f[1]
y = plt_data['v_phi_major_at_ISM_1o5']
ax.fmt_marker(**axis_kw['major']).scatter(x, y, **plt_kws['ISM'])
y = plt_data['v_phi_major_at_CGM_half']
ax.fmt_marker(**axis_kw['minor']).scatter(x, y, **plt_kws['CGM'])

x1, y1 = draw_scaling(x_tick_lim, y0=4.5, gamma=1./3.)
ax.plot(x1, y1, lw=1.5, ls=(0,(2,2)), c='gray')
ax.lim(y=[1.0, 755]).label(y=r'v_{\phi}\, [{\rm km/s}]')

ax = axs_f[2]
y = plt_data['lambda_major_at_ISM_1o5']
ax.fmt_marker(**axis_kw['major']).scatter(x, y, **plt_kws['ISM'])
y = plt_data['lambda_major_at_CGM_half']
ax.fmt_marker(**axis_kw['minor']).scatter(x, y, **plt_kws['CGM'])

x1, y1 = draw_scaling(x_tick_lim, y0=0.03, gamma=0.)
ax.plot(x1, y1, lw=1.5, ls=(0,(2,2)), c='gray')
ax.lim(y=[0.01, 0.3]).label(y=r'\lambda')
    
ax = axs_f[3]
y = 10.0**plt_data['lg_T_major_at_ISM_1o5']
ax.fmt_marker(**axis_kw['major']).scatter(x, y, **plt_kws['ISM'])
y = 10.0**plt_data['lg_T_minor_at_ISM_1o5']
ax.fmt_marker(c=cs_named['r'], fa=.3, elw=2).scatter(x, y, **plt_kws['ISM'])

y = 10.0**plt_data['lg_T_major_at_CGM_half']
ax.fmt_marker(**axis_kw['major']).scatter(x, y, **plt_kws['CGM'])
y = 10.0**plt_data['lg_T_minor_at_CGM_half']
ax.fmt_marker(c=cs_named['r'], fa=.3, elw=2).scatter(x, y, **plt_kws['CGM'])

x1, y1 = draw_scaling(x_tick_lim, y0=10**4.55, gamma=2./3.)
ax.plot(x1, y1, lw=1.5, ls=(0,(2,2)), c='gray')
ax.lim(y=[2.0e3, 10**6.95])\
    .label(y=r'T\, [{\rm K}]')

ax = axs_f[4]
y = -plt_data['v_r_major_at_ISM_1o5']
ax.fmt_marker(**axis_kw['major']).scatter(x, y, **plt_kws['ISM'])
y = plt_data['v_r_minor_at_ISM_1o5']
ax.fmt_marker(c=cs_named['r'], fa=.3, elw=2).scatter(x, y, **plt_kws['ISM'])

y = -plt_data['v_r_major_at_CGM_half']
ax.fmt_marker(**axis_kw['major']).scatter(x, y, **plt_kws['CGM'])
y = plt_data['v_r_minor_at_CGM_half']
ax.fmt_marker(c=cs_named['r'], fa=.3, elw=2).scatter(x, y, **plt_kws['CGM'])

x1, y1 = draw_scaling(x_tick_lim, y0=10**2.15, gamma=1.)
ax.plot(x1, y1, lw=1.5, ls=(0,(2,2)), c='gray')
ax.lim(y=[.1, 3.0e3]).label(y=r'\left|v_r\right|\, [{\rm km/s}]')

ax = axs_f[5]
y = -plt_data['I_major_at_ISM_1o5']
ax.fmt_marker(**axis_kw['major']).scatter(x, y, **plt_kws['ISM'])
y = plt_data['I_minor_at_ISM_1o5']
ax.fmt_marker(c=cs_named['r'], fa=.3, elw=2).scatter(x, y, **plt_kws['ISM'])

y = -plt_data['I_major_at_CGM_half']
ax.fmt_marker(**axis_kw['major']).scatter(x, y, **plt_kws['CGM'])
y = plt_data['I_minor_at_CGM_half']
ax.fmt_marker(c=cs_named['r'], fa=.3, elw=2).scatter(x, y, **plt_kws['CGM'])

x1, y1 = draw_scaling(x_tick_lim, y0=.5, gamma=4./3.)
ax.plot(x1, y1, lw=1.5, ls=(0,(2,2)), c='gray')
ax.lim(y=[.01, 1e1]).label(y=r'\left|I\right|\, [{\rm M_\odot yr^{-1} sr^{-1}}]')
    

axs.lim([10**10.5, 10**12.6]).scale('log', 'log').label(r'M_{\rm h}\, [{\rm M}_\odot]')
axs[0].label(r'\,')

plot.savefig(out_fig_dir/'role-of-halo-gas.pdf')

In [ ]:
fig, ax = plot.subplots(1, figsize=5, margin=[0.02, 0.02, 0.1, 0.135], layout='none')

x = plt_data['M_h']
y = plt_data['R_ISM']
ax.fmt_marker(c='k', fa=.3, elw=2).scatter(x, y, **plt_kws['ISM'])
y = plt_data['R_CGM']
ax.fmt_marker(c='k', fa=.3, elw=2).scatter(x, y, **plt_kws['CGM'])

x1, y1 = draw_scaling([2e11, 8e11], y0=15, gamma=1./3.)
ax.plot(x1, y1, lw=1.5, ls=(0,(2,2)), c='gray')
ax.lim(y=[7.5, 1.0e3]).label(y=r'R\, [{\rm kpc}]')

ax.lim([10**10.5, 10**12.6]).scale('log', 'log').label(r'M_{\rm h}\, [{\rm M}_\odot]')

plot.savefig(out_fig_dir/'role-of-halo-scales.pdf')

In [ ]:
r_edges = all_data['header']['r_edges']
v_phi = all_data['z0/lm10']['v_phi/major']

fig, ax = plot.subplots(1, figsize=4.5, margin=[0.1, 0.1, 0.1, 0.1], layout='none')
ax.plot(r_edges[:-1], v_phi)

ax.scale('log')